# Intro  

https://python.langchain.com/docs/concepts/rag/  

https://python.langchain.com/docs/how_to/#qa-with-rag  

https://python.langchain.com/docs/tutorials/  

https://python.langchain.com/docs/tutorials/rag/  

https://wikidocs.net/231393  

# Google Gemini Example  

In [ ]:
import google.generativeai as genai
import pathlib
import os
import time
import textwrap
from IPython.display import display
from IPython.display import Markdown

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Gemini API Key  
API_KEY = "YOUR-API-KEY"

# Used to securely store your API key
#from google.colab import userdata
# Or use `os.getenv('GOOGLE_API_KEY')` to fetch an environment variable.
#GOOGLE_API_KEY=userdata.get(API_KEY)

genai.configure(api_key=API_KEY)

# Credentials 관련 조치
credentials_path = "credentials.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
time.sleep(3)  # Let the environment variable propagate

/home/taehan/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-1.0-pro-latest
models/gemini-1.0-pro
models/gemini-pro
models/gemini-1.0-pro-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/learnlm-1.5-pro-experimental


# Load Gemini model

In [3]:
model = genai.GenerativeModel('gemini-1.5-flash')
model

genai.GenerativeModel(
    model_name='models/gemini-1.5-flash',
    generation_config={},
    safety_settings={},
    tools=None,
    system_instruction=None,
    cached_content=None
)

# LangChain RAG  

## RAG Steps  

1. Load Documents  

2. Split  
Splitter breaks large text into smaller chunks.  
This is useful both for indexing data and passing it into a model,  
as large chunks are harder to search over and won't fit in a model's finite context window.  

3. Store - Vector store and Embeddings  
We need somewhere to store and index our splits, so that they can be searched over later.  
This is often done using a VectorStore and Embeddings model.  

4. Retrieve  
Given a user input, relevant splits are retrieved from storage using a Retriever.  

5. Generate  
A ChatModel / LLM produces an answer using a prompt that includes both the question with the retrieved data.  

Once we've indexed our data, we will use LangGraph as our orchestration framework to implement the retrieval and generation steps.  


https://python.langchain.com/docs/how_to/#document-loaders  

https://python.langchain.com/docs/tutorials/rag/  

+ Document loaders  
pdf  
csv  
web pages  
html  
json  
markdown  
ms office  

+ Text splitters  
split html  
split json  
split code  
split by character  

+ Embedding models  
embed text  
cache embedding  

+ Vector stores  
retrieve data  

+ Retrievers  

## 1. Load Documents  

In [4]:
import os
import pathlib

doc_path = 'document'

os.listdir(doc_path)

['ETF-GLD-20240331 SPDR.pdf',
 'ETF-GLD-Key-info SPDR.pdf',
 'etf_info.txt',
 'factsheet-us-en-splg.pdf',
 'factsheet-us-en-spy.pdf',
 'Invesco QQQ ETF - Infographic.pdf',
 'ishares-core-us-stock-product-brief-itot-ivv-ijh-ijr-en-us.pdf',
 'ivv-ishares-core-s-p-500-etf-fund-fact-sheet-en-us.pdf',
 'KODEX_S&P500_ETF_20250131.pdf',
 'KR7360750004_R_TIGER_미국S&P500_2024-12-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_2025-01-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_간이투자설명서.pdf',
 'KR7360750004_R_TIGER_미국S&P500_투자설명서.pdf',
 'p-ishares-core-s-and-p-500-etf-3-31.pdf',
 'p-ishares-silver-trust-prospectus-12-31.pdf',
 'P-QQQ-AR-1.pdf',
 'P-QQQ-PRO-1.pdf',
 'P-QQQM-AR.pdf',
 'P-QQQM-PRO-1.pdf',
 'P-TRST2-SAR-1.pdf',
 'QQQ - Invesco QQQ ETF fact sheet.pdf',
 'QQQ Insight PowerShares Invesco.pdf',
 'QQQM - Invesco NASDAQ 100 ETF - Infographic.pdf',
 'QQQM - Invesco NASDAQ 100 ETF fact sheet.pdf',
 'SCHH - Schwab US REIT ETF Factsheet.pdf',
 'SCHH - Schwab US REIT ETF performance summary.pdf',
 'SCHH

### SPY Example  

In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = os.path.join(doc_path, 'factsheet-us-en-spy.pdf')

loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [6]:
print(f"{pages[0].metadata}\n")
print(pages[0].page_content)

{'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}

1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment 
results that, before expenses, correspond generally to 
the price and yield performance of the S&P 500® Index 
(the “Index”)
 • The S&P 500 Index is a diversified large cap U.S. index that 
holds companies across all eleven GICS sectors
 • Launched in January 1993, SPY was the very first exchange 
traded fund listed in the United States
About This Benchmark
The S&P 500® Index is designed to measure the performance of 
the large-cap segment of the US equity market. It is float-adjusted 
market capitalization weighted. 
Fund Information
Inception Date 01/22/1993
CUSIP 78462F103
T otal Return (As of 12/31/2024)
 NAV
(%)
Market Value
(%)
Index
(%)
Cumulative    
QTD 2.38 2.43 2.41
YTD 24.87 24.86 25.02
Annualized    
1 Year 24.87 24.86 25.02
3 Year 8.81 8.81 8.94
5 Year 14.38 14.40 14.53
1

In [7]:
pages

[Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\nYTD 24.87 24.86 25.02\nAnnualized    \n1 Ye

### Gemini Embedding  

https://python.langchain.com/api_reference/google_genai/embeddings/langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings.html#langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
embeddings = embeddings_model.embed_query("What's our Q1 revenue?")

In [9]:
print(f'type of embeddings: {type(embeddings)}')

type of embeddings: <class 'list'>


In [10]:
print(f"length of embeddings: {len(embeddings)}")
embeddings[0:10]

length of embeddings: 768


[0.01612485572695732,
 -0.03933568298816681,
 -0.002993757603690028,
 -0.033227454870939255,
 0.05022253096103668,
 0.03145093098282814,
 -0.00027826917357742786,
 -0.064089834690094,
 0.05403633043169975,
 0.02948707528412342]

Length of embeddings is 768.  

## 2. Splitter  

spliter.split_documents에 들어갈 input documents의 형식  

```
documents = [
    {"text": "This is the first document. It contains some text.", "metadata": {"source": "source1"}},
    {"text": "This is the second document. It also contains some text.", "metadata": {"source": "source2"}},
    # 추가 문서들...
]
```

https://python.langchain.com/docs/how_to/recursive_text_splitter/  

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
'''
# Load example document
with open("state_of_the_union.txt") as f:
    state_of_the_union = f.read()

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
texts = text_splitter.create_documents([state_of_the_union])
print(texts[0])
print(texts[1])
'''

'\n# Load example document\nwith open("state_of_the_union.txt") as f:\n    state_of_the_union = f.read()\n\ntext_splitter = RecursiveCharacterTextSplitter(\n    # Set a really small chunk size, just to show.\n    chunk_size=100,\n    chunk_overlap=20,\n    length_function=len,\n    is_separator_regex=False,\n)\ntexts = text_splitter.create_documents([state_of_the_union])\nprint(texts[0])\nprint(texts[1])\n'

### Length of Input Token in Gemini-1.5-pro 

Input size: 2,097,152  
Output size: 8,192  

https://ai.google.dev/gemini-api/docs/models/gemini?hl=ko#rate-limits  

### Small Sized Chunk    

Even though Gemini-1.5-pro supports long length,  
set very small chunk size for making an example.  

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)


In [13]:
### Create Documents  

print(f"length of pages: {len(pages)}")
page_index = 0
current_page = pages[page_index].page_content
print(f"Current page: {page_index}")

texts = text_splitter.create_documents([current_page])

print(f"The number of chunk: {len(texts)}\n")
print("texts 0:\n",texts[0])
print("texts 1:\n",texts[1])

length of pages: 2
Current page: 0
The number of chunk: 29

texts 0:
 page_content='1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features'
texts 1:
 page_content='Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment'


In [14]:
texts

[Document(metadata={}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features'),
 Document(metadata={}, page_content='Key Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment'),
 Document(metadata={}, page_content='results that, before expenses, correspond generally to'),
 Document(metadata={}, page_content='the price and yield performance of the S&P 500® Index \n(the “Index”)'),
 Document(metadata={}, page_content='(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that'),
 Document(metadata={}, page_content='holds companies across all eleven GICS sectors'),
 Document(metadata={}, page_content='• Launched in January 1993, SPY was the very first exchange'),
 Document(metadata={}, page_content='traded fund listed in the United States\nAbout This Benchmark'),
 Document(metadata={}, page_content='The S&P 500® Index is designed to measure the performance of'),
 Document(metadata={}, page_content='th

In [15]:
### Split Documents  

# Input should Document object
all_splits = text_splitter.split_documents(pages)
all_splits

[Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features'),
 Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='Key Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment'),
 Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='results that, before expenses, correspond generally to'),
 Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='the price and yield performance of the S&P 500® Index \n(the “Index”)'),
 Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that'),
 Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='holds companies across all eleven GICS sectors'),
 Document(me

### Large Sized Chunk  

Set chunk size as the half Gemini-1.5-flash's input sequence length.  

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(2097152 / 2)
chunk_overlap_size = int(chunk_size/10) # 10% of chunk_size

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap_size,
    length_function=len,
    is_separator_regex=False,
)


In [17]:
### Create Documents  

print(f"length of pages: {len(pages)}")
page_index = 0
current_page = pages[page_index].page_content
print(f"Current page: {page_index}")

texts = text_splitter.create_documents([current_page])

print(f"The number of chunk: {len(texts)}\n")
print("texts 0:\n",texts[0])

length of pages: 2
Current page: 0
The number of chunk: 1

texts 0:
 page_content='1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment 
results that, before expenses, correspond generally to 
the price and yield performance of the S&P 500® Index 
(the “Index”)
 • The S&P 500 Index is a diversified large cap U.S. index that 
holds companies across all eleven GICS sectors
 • Launched in January 1993, SPY was the very first exchange 
traded fund listed in the United States
About This Benchmark
The S&P 500® Index is designed to measure the performance of 
the large-cap segment of the US equity market. It is float-adjusted 
market capitalization weighted. 
Fund Information
Inception Date 01/22/1993
CUSIP 78462F103
T otal Return (As of 12/31/2024)
 NAV
(%)
Market Value
(%)
Index
(%)
Cumulative    
QTD 2.38 2.43 2.41
YTD 24.87 24.86 25.02
Annualized    
1 Year 24.87 24.86 25.02
3 Year 8.81 8.81 8.94
5 

In [18]:
texts

[Document(metadata={}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\nYTD 24.87 24.86 25.02\nAnnualized    \n1 Year 24.87 24.86 25.02\n3 Year 8.81 8.81 8.94\n5 Year 14.

### Split Each Pages in a Document  

In [19]:
### Split Documents  

# Input should Document object
all_splits = text_splitter.split_documents(pages)
all_splits

[Document(metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\nYTD 24.87 24.86 25.02\nAnnualized    \n1 Ye

## 3. Vector Store, Embeddings, and Retriever   

+ Embedding  
https://python.langchain.com/docs/how_to/embed_text/  

+ Vector Store  
https://python.langchain.com/docs/how_to/vectorstores/  



### Google Vector Store   

Currently, it computes the embedding vectors on the server side.  



https://python.langchain.com/api_reference/google_genai/google_vector_store/langchain_google_genai.google_vector_store.GoogleVectorStore.html#langchain_google_genai.google_vector_store.GoogleVectorStore

In [20]:
from langchain_google_genai.google_vector_store import GoogleVectorStore

# add texts to an existing corpus.
#store = GoogleVectorStore(corpus_id='123').store.add_documents(documents, document_id="456")

### Google Vertex AI Vector Search  

https://python.langchain.com/docs/integrations/vectorstores/google_vertex_ai_vector_search/  

### FAISS  

https://python.langchain.com/docs/integrations/vectorstores/faiss/  

https://wikidocs.net/234014  

https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html  

In [21]:
from langchain_community.vectorstores import FAISS

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

vectorstore = FAISS.from_documents(all_splits, embeddings_model)

retriever = vectorstore.as_retriever()

In [22]:
docs = retriever.invoke("Explain S&P 500 ETF")
docs

[Document(id='bb37c935-c892-4682-9670-2b9dfc2c36bd', metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\n

In [23]:
# Maximum marginal relevance retrieval
retriever = vectorstore.as_retriever(search_type="mmr")
docs = retriever.invoke("Explain S&P 500 ETF")
docs

[Document(id='bb37c935-c892-4682-9670-2b9dfc2c36bd', metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\n

In [24]:
# Similarity score threshold retrieval

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
)

docs = retriever.invoke("Explain S&P 500 ETF")
docs

[Document(id='bb37c935-c892-4682-9670-2b9dfc2c36bd', metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\n

In [25]:
# Specifying top k

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke("Explain S&P 500 ETF")
print(f"len(docs): {len(docs)}")
docs

len(docs): 2


[Document(id='bb37c935-c892-4682-9670-2b9dfc2c36bd', metadata={'source': 'document/factsheet-us-en-spy.pdf', 'page': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.38 2.43 2.41\n

### In Memory Vector Store    

https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.in_memory.InMemoryVectorStore.html  

In [26]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings_model)

In [27]:
# Index chunks
document_ids = vector_store.add_documents(documents=all_splits)

# add_documents method returns document_ids in vector store
print(document_ids[:3])

['e66dd3dd-1ce8-4141-a2fc-85e233d3cecd', 'f52b5d92-9642-4a05-923b-3a22644ab7f8']


In [28]:
results = vector_store.similarity_search(query="Explain S&P 500 ETF",k=1)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* 1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment 
results that, before expenses, correspond generally to 
the price and yield performance of the S&P 500® Index 
(the “Index”)
 • The S&P 500 Index is a diversified large cap U.S. index that 
holds companies across all eleven GICS sectors
 • Launched in January 1993, SPY was the very first exchange 
traded fund listed in the United States
About This Benchmark
The S&P 500® Index is designed to measure the performance of 
the large-cap segment of the US equity market. It is float-adjusted 
market capitalization weighted. 
Fund Information
Inception Date 01/22/1993
CUSIP 78462F103
T otal Return (As of 12/31/2024)
 NAV
(%)
Market Value
(%)
Index
(%)
Cumulative    
QTD 2.38 2.43 2.41
YTD 24.87 24.86 25.02
Annualized    
1 Year 24.87 24.86 25.02
3 Year 8.81 8.81 8.94
5 Year 14.38 14.40 14.53
10 Year 12.96 12.96 13.10
  
Gross Expense Ratio (%) 0.094

In [29]:
# Search with score  

results = vector_store.similarity_search_with_score(
    query="Explain S&P 500 ETF", k=1
)

for doc, score in results:
    print(f"* [SIM={score:3f}] {doc.page_content} [{doc.metadata}]")

* [SIM=0.723567] 1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment 
results that, before expenses, correspond generally to 
the price and yield performance of the S&P 500® Index 
(the “Index”)
 • The S&P 500 Index is a diversified large cap U.S. index that 
holds companies across all eleven GICS sectors
 • Launched in January 1993, SPY was the very first exchange 
traded fund listed in the United States
About This Benchmark
The S&P 500® Index is designed to measure the performance of 
the large-cap segment of the US equity market. It is float-adjusted 
market capitalization weighted. 
Fund Information
Inception Date 01/22/1993
CUSIP 78462F103
T otal Return (As of 12/31/2024)
 NAV
(%)
Market Value
(%)
Index
(%)
Cumulative    
QTD 2.38 2.43 2.41
YTD 24.87 24.86 25.02
Annualized    
1 Year 24.87 24.86 25.02
3 Year 8.81 8.81 8.94
5 Year 14.38 14.40 14.53
10 Year 12.96 12.96 13.10
  
Gross Expense 

### FAISS and InMemoryVectorStore  

https://python.langchain.com/docs/integrations/vectorstores/faiss/  

https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html  


In [30]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

index = faiss.IndexFlatL2(len(embeddings_model.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)


In [31]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocalate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['951ea865-1e55-4458-ad97-ef7e94e5f671',
 'efbdf063-cee0-4591-8cca-4877237e9990',
 '9cbc87d1-3d28-429c-9872-d49555c1b89c',
 '4dc577bb-9ccc-4c09-aeb6-0e4d5f2cda22',
 '6318c19e-bf0d-4856-9f9d-e4a0df81ca29',
 '628a0745-167a-479e-b662-519bbc58dbf7',
 '63e31c2a-9a46-424d-a53f-05af8e4d9ba6',
 'd9a288f0-c9c9-41e1-8960-ccf5c06f8cf4',
 'b5679d4d-e3e7-44e0-95b8-f8a41e9a6091',
 'f06ce728-b988-4798-b823-64c6fe59e211']

In [32]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]
* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]


In [33]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.664568] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


## 4. Retrieve  

https://python.langchain.com/docs/how_to/vectorstore_retriever/  
https://python.langchain.com/docs/concepts/retrieval/  


In [34]:
# Declare retriever
from langchain_community.vectorstores import FAISS

# Embedding Models  
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# FAISS Vector Store
vectorstore = FAISS.from_documents(all_splits, embeddings_model)

retriever = vectorstore.as_retriever()

### Split Documents  

In [35]:
# Split Documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(2097152 / 2)
chunk_overlap_size = int(chunk_size/10) # 10% of chunk_size

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap_size,
    length_function=len,
    is_separator_regex=False,
)


In [36]:
import os
doc_path = 'document'

document_names_list = os.listdir(doc_path)
document_names_list

['ETF-GLD-20240331 SPDR.pdf',
 'ETF-GLD-Key-info SPDR.pdf',
 'etf_info.txt',
 'factsheet-us-en-splg.pdf',
 'factsheet-us-en-spy.pdf',
 'Invesco QQQ ETF - Infographic.pdf',
 'ishares-core-us-stock-product-brief-itot-ivv-ijh-ijr-en-us.pdf',
 'ivv-ishares-core-s-p-500-etf-fund-fact-sheet-en-us.pdf',
 'KODEX_S&P500_ETF_20250131.pdf',
 'KR7360750004_R_TIGER_미국S&P500_2024-12-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_2025-01-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_간이투자설명서.pdf',
 'KR7360750004_R_TIGER_미국S&P500_투자설명서.pdf',
 'p-ishares-core-s-and-p-500-etf-3-31.pdf',
 'p-ishares-silver-trust-prospectus-12-31.pdf',
 'P-QQQ-AR-1.pdf',
 'P-QQQ-PRO-1.pdf',
 'P-QQQM-AR.pdf',
 'P-QQQM-PRO-1.pdf',
 'P-TRST2-SAR-1.pdf',
 'QQQ - Invesco QQQ ETF fact sheet.pdf',
 'QQQ Insight PowerShares Invesco.pdf',
 'QQQM - Invesco NASDAQ 100 ETF - Infographic.pdf',
 'QQQM - Invesco NASDAQ 100 ETF fact sheet.pdf',
 'SCHH - Schwab US REIT ETF Factsheet.pdf',
 'SCHH - Schwab US REIT ETF performance summary.pdf',
 'SCHH

In [37]:
from langchain_community.document_loaders import PyPDFLoader

document_name = document_names_list[0]
print(f'Document : {document_name}')
file_path = os.path.join(doc_path, document_name)

loader = PyPDFLoader(file_path)
pages = []
raw_texts = []
async for page in loader.alazy_load():
    pages.append(page)
    raw_texts.append(page.page_content)

print(f"Total {len(pages)} pages")

Document : ETF-GLD-20240331 SPDR.pdf
Total 2 pages


In [38]:
### Create Split Documents  

texts = text_splitter.create_documents(raw_texts)

n_chunks = len(texts)
print(f"The number of chunk: {n_chunks}\n")
print("texts 0:\n",texts[0].page_content[0:200],'...Trimmed...\n')
print(f"texts {n_chunks - 1}:\n",texts[n_chunks - 1].page_content[0:200],'...Trimmed...')

The number of chunk: 2

texts 0:
 SPDR® Gold Shares GLD®
 
Fact Sheet
Gold
 
As of 03/31/2024
1
Objective
The investment objective of the Trust is for SPDR® Gold Shares 
(GLD®) to reflect the performance of the price of gold bullion,  ...Trimmed...

texts 1:
 2
Advantages
Easily Accessible Listed on the NYSE Arca.
Secure The Gold Shares represent fractional, undivided 
interests in the Trust, the sole assets of which are 
physical gold bullion and, from ti ...Trimmed...


In [39]:
document_names_list_orig = document_names_list
document_names_list = []

for document_name in document_names_list_orig:
    if document_name.endswith('.pdf'):
        document_names_list.append(document_name)

document_names_list

['ETF-GLD-20240331 SPDR.pdf',
 'ETF-GLD-Key-info SPDR.pdf',
 'factsheet-us-en-splg.pdf',
 'factsheet-us-en-spy.pdf',
 'Invesco QQQ ETF - Infographic.pdf',
 'ishares-core-us-stock-product-brief-itot-ivv-ijh-ijr-en-us.pdf',
 'ivv-ishares-core-s-p-500-etf-fund-fact-sheet-en-us.pdf',
 'KODEX_S&P500_ETF_20250131.pdf',
 'KR7360750004_R_TIGER_미국S&P500_2024-12-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_2025-01-31 .pdf',
 'KR7360750004_R_TIGER_미국S&P500_간이투자설명서.pdf',
 'KR7360750004_R_TIGER_미국S&P500_투자설명서.pdf',
 'p-ishares-core-s-and-p-500-etf-3-31.pdf',
 'p-ishares-silver-trust-prospectus-12-31.pdf',
 'P-QQQ-AR-1.pdf',
 'P-QQQ-PRO-1.pdf',
 'P-QQQM-AR.pdf',
 'P-QQQM-PRO-1.pdf',
 'P-TRST2-SAR-1.pdf',
 'QQQ - Invesco QQQ ETF fact sheet.pdf',
 'QQQ Insight PowerShares Invesco.pdf',
 'QQQM - Invesco NASDAQ 100 ETF - Infographic.pdf',
 'QQQM - Invesco NASDAQ 100 ETF fact sheet.pdf',
 'SCHH - Schwab US REIT ETF Factsheet.pdf',
 'SCHH - Schwab US REIT ETF performance summary.pdf',
 'SCHH - Schwab US REIT

In [40]:
from langchain_community.document_loaders import PyPDFLoader

from uuid import uuid4

document_name = document_names_list[0]
print(f'Document : {document_name}')

def get_document_path(doc_path, document_name):
    return os.path.join(doc_path, document_name)

file_path = get_document_path(doc_path, document_name)

# async function  
async def load_pdf_file(file_path):
    loader = PyPDFLoader(file_path)
    pages = []
    raw_texts = []
    async for page in loader.alazy_load():
        pages.append(page)
        raw_texts.append(page.page_content)
        #raw_texts.extend(page.page_content)
    return pages, raw_texts

async def load_document(doc_path, document_name):
    file_path = get_document_path(doc_path, document_name)
    if document_name.endswith('.pdf'):
        return await load_pdf_file(file_path)
    elif document_name.endswith('.txt'):
        pass
    else:
        raise ValueError(f"Unsupported file type: {document_name}")
    
### Create Split Texts from Current Document  

async def create_split_texts(doc_path, document_name):
    pages, raw_texts = await load_document(doc_path, document_name)
    print(f"raw text: {type(raw_texts)} \n{raw_texts}")
    print(f"Total {len(pages)} pages")
    '''
    texts = text_splitter.create_documents(texts = raw_texts, 
                                           metadatas=[{"document_name": document_name, 'doc_id': str(uuid4())}])
    n_chunks = len(texts)
    '''
    chunks = text_splitter.split_text(raw_texts[0])
    print(f"chunks: \n{chunks}")
    n_chunks = len(chunks)
    # 각 청크마다 새로운 메타데이터 생성
    metadatas = [{'document_id': str(uuid4()), "chunk_id": i} for i in range(n_chunks)]
    texts = text_splitter.create_documents(chunks, metadatas)
    print(f"The number of chunk: {n_chunks}\n")
    print("texts 0:\n",texts[0].page_content[0:200],'...Trimmed...\n')
    print(f"texts {n_chunks - 1}:\n",texts[n_chunks - 1].page_content[0:200],'...Trimmed...')
    return texts


Document : ETF-GLD-20240331 SPDR.pdf


In [41]:
split_documents = []

for document_name in document_names_list:
    print(f'Document : {document_name}')
    texts = await create_split_texts(doc_path, document_name)
    split_documents.extend(texts)

Document : ETF-GLD-20240331 SPDR.pdf
raw text: <class 'list'> 
['SPDR® Gold Shares GLD®\n \nFact Sheet\nGold\n \nAs of 03/31/2024\n1\nObjective\nThe investment objective of the Trust is for SPDR® Gold Shares \n(GLD®) to reflect the performance of the price of gold bullion, less \nthe Trust’s expenses.\nThe Price of Gold\nThe spot price for gold bullion is determined by market forces in \nthe 24-hour global over-the-counter (OTC) market for gold. The \nOTC market accounts for most global gold trading, and prices \nquoted reflect the information available to the market at any given \ntime. The price, holdings, and net asset value of the Gold Shares, \nas well as market data for the overall gold bullion market, can be \ntracked daily at spdrgoldshares.com.\nFund Information\nInception Date 11/18/2004\nIntraday NAV Ticker GLDIV\nIndex Ticker N/A\nKey Facts\nTicker Symbol GLD®\nCUSIP 78463V107\nExchange NYSE ARCA EXCHANGE\nShort Sale Eligible Ye s\nMargin Eligible Ye s\nT otal Return (As of

Document object  

https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html

In [42]:
from langchain_core.documents import Document

document = Document(
    page_content="Hello, world!",
    metadata={"source": "https://example.com"}
)

In [44]:
split_documents[0], type(split_documents[0])

(Document(metadata={'document_id': '06136c1c-d392-499a-a62c-5a89f228dcde', 'chunk_id': 0}, page_content='SPDR® Gold Shares GLD®\n \nFact Sheet\nGold\n \nAs of 03/31/2024\n1\nObjective\nThe investment objective of the Trust is for SPDR® Gold Shares \n(GLD®) to reflect the performance of the price of gold bullion, less \nthe Trust’s expenses.\nThe Price of Gold\nThe spot price for gold bullion is determined by market forces in \nthe 24-hour global over-the-counter (OTC) market for gold. The \nOTC market accounts for most global gold trading, and prices \nquoted reflect the information available to the market at any given \ntime. The price, holdings, and net asset value of the Gold Shares, \nas well as market data for the overall gold bullion market, can be \ntracked daily at spdrgoldshares.com.\nFund Information\nInception Date 11/18/2004\nIntraday NAV Ticker GLDIV\nIndex Ticker N/A\nKey Facts\nTicker Symbol GLD®\nCUSIP 78463V107\nExchange NYSE ARCA EXCHANGE\nShort Sale Eligible Ye s\nMa

In [45]:
split_documents[1]

Document(metadata={'document_id': 'fd293f49-9fbe-4b90-9853-32e7ee1efe07', 'chunk_id': 0}, page_content='Key Information GLD\nSPDR® GOLD SHARES\nOBJECTIVE Designed to track the price of gold (net of Trust expenses).  See Important Risk Disclosures below regarding the \nrisk of investing in GLD.\nSTRUCTURE Continuously offered investment trust \nSYMBOL GLD\nEXCHANGE NYSE Arca, Inc.\nINITIAL PRICING Based on the price of 1/10th of an ounce of gold\nMINIMUM ORDER 1 share\nSHORT SALE ELIGIBLE Yes\nMARGIN ELIGIBLE Yes\nESTIMATED EXPENSES 0.40%*\nGOLD BULLION\nALLOCATED GOLD The Trust’s gold bullion is kept in the form of London Good Delivery bars (400 oz.) and held in an allocated \naccount.**\nSTORAGE The gold bullion is held by the Custodian, HSBC Bank USA, in its London vault or in the vaults of sub-custodians.\nADVANTAGES\nEASILY ACCESSIBLE Listed on the NYSE Arca\nSECURE The Gold Shares represent fractional, undivided interests in the Trust, the primary asset of which is allocated (or \

In [46]:
dir(split_documents[0])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__',
 '__pydantic_validator__',

In [47]:
split_documents[0].id, split_documents[0].lc_id

(None,
 <bound method Serializable.lc_id of <class 'langchain_core.documents.base.Document'>>)

### Sparse Retrieve  

Traditional search method  

BM25  

BM25Retriever is [Runnable Interface](https://api.python.langchain.com/en/latest/core/runnables/langchain_core.runnables.base.Runnable.html#langchain_core.runnables.base.Runnable).   

https://api.python.langchain.com/en/latest/community/retrievers/langchain_community.retrievers.bm25.BM25Retriever.html  

https://python.langchain.com/docs/integrations/retrievers/bm25/  

https://python.langchain.com/api_reference/_modules/langchain_community/retrievers/bm25.html#BM25Retriever.from_documents   

https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/retrievers/bm25.py  

Rank BM25  
https://github.com/dorianbrown/rank_bm25  

In [48]:
#!pip install rank_bm25

from langchain.retrievers import BM25Retriever

# BM25 인덱스 생성
'''Data structure of split_documents as below
[
        Document(page_content="foo"),
        Document(page_content="bar"),
        Document(page_content="world"),
        Document(page_content="hello"),
        Document(page_content="foo bar"),
    ]
)
'''

bm25_retriever = BM25Retriever.from_documents(split_documents)


In [49]:
# invoke methods is equivalent to get_relevant_documents

query = "Explain S&P 500 ETF"
bm25_results = bm25_retriever.invoke(query)
bm25_results


[Document(metadata={'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}, page_content='SPDR S&P 500 \nETF Trust (SPY) \nDelivering Unrivaled \nLiquidity to Investors'),
 Document(metadata={'document_id': '036658bb-b498-4d1d-8078-0a7aab520669', 'chunk_id': 0}, page_content='IVV\niShares Core S&P 500 ETF\nFact Sheet as of 31-Dec-2024\nThe iShares Core S&P 500 ETF seeks to track the investment results of an index \ncomposed of large-capitalization U.S. equities.\nWHY IVV?\n1 Exposure to large, established U.S. companies\n2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks\n3 Use at the core of your portfolio to seek long-term growth\nGROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION\nFund  Benchmark  \nThe Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes \nreinvestment of dividends and capital gains. Fund expenses, including management fees and \nother expenses were deducted.\nPERFORMANCE\n1 Year 3 Year 5 Year 10 Year Sinc

In [50]:
for result in bm25_results:
    print(f"* {result.page_content} [{result.metadata}]")

* SPDR S&P 500 
ETF Trust (SPY) 
Delivering Unrivaled 
Liquidity to Investors [{'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}]
* IVV
iShares Core S&P 500 ETF
Fact Sheet as of 31-Dec-2024
The iShares Core S&P 500 ETF seeks to track the investment results of an index 
composed of large-capitalization U.S. equities.
WHY IVV?
1 Exposure to large, established U.S. companies
2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks
3 Use at the core of your portfolio to seek long-term growth
GROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION
Fund  Benchmark  
The Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes 
reinvestment of dividends and capital gains. Fund expenses, including management fees and 
other expenses were deducted.
PERFORMANCE
1 Year 3 Year 5 Year 10 Year Since Inception
NAV 24.98% 8.91% 14.49% 13.06% 7.79%
Market Price 24.94% 8.91% 14.51% 13.06% 7.79%
Benchmark 25.02% 8.94% 14.53% 13.10% 7.85%
The  performance

LangChain의 BM25 Retriever의 [내부 코드](https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/retrievers/bm25.py)를 살펴보면 rank_bm25에서 BM25Okapi를 사용함을 알 수 있다.  

```
try:
            from rank_bm25 import BM25Okapi
        except ImportError:
            raise ImportError(
                "Could not import rank_bm25, please install with `pip install "
                "rank_bm25`."
            )
```

[rank_bm25](https://github.com/dorianbrown/rank_bm25/blob/master/rank_bm25.py)의 BM25Okapi 클래스의 코드를 살펴보면 다음과 같다.  

```
class BM25Okapi(BM25):
    def __init__(self, corpus, tokenizer=None, k1=1.5, b=0.75, epsilon=0.25):
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon
        super().__init__(corpus, tokenizer)

    def _calc_idf(self, nd):
        """
        Calculates frequencies of terms in documents and in corpus.
        This algorithm sets a floor on the idf values to eps * average_idf
        """
        # collect idf sum to calculate an average idf for epsilon value
        idf_sum = 0
        # collect words with negative idf to set them a special epsilon value.
        # idf can be negative if word is contained in more than half of documents
        negative_idfs = []
        for word, freq in nd.items():
            idf = math.log(self.corpus_size - freq + 0.5) - math.log(freq + 0.5)
            self.idf[word] = idf
            idf_sum += idf
            if idf < 0:
                negative_idfs.append(word)
        self.average_idf = idf_sum / len(self.idf)

        eps = self.epsilon * self.average_idf
        for word in negative_idfs:
            self.idf[word] = eps

    def get_scores(self, query):
        """
        The ATIRE BM25 variant uses an idf function which uses a log(idf) score. To prevent negative idf scores,
        this algorithm also adds a floor to the idf value of epsilon.
        See [Trotman, A., X. Jia, M. Crane, Towards an Efficient and Effective Search Engine] for more info
        :param query:
        :return:
        """
        score = np.zeros(self.corpus_size)
        doc_len = np.array(self.doc_len)
        for q in query:
            q_freq = np.array([(doc.get(q) or 0) for doc in self.doc_freqs])
            score += (self.idf.get(q) or 0) * (q_freq * (self.k1 + 1) /
                                               (q_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)))
        return score

    def get_batch_scores(self, query, doc_ids):
        """
        Calculate bm25 scores between query and subset of all docs
        """
        assert all(di < len(self.doc_freqs) for di in doc_ids)
        score = np.zeros(len(doc_ids))
        doc_len = np.array(self.doc_len)[doc_ids]
        for q in query:
            q_freq = np.array([(self.doc_freqs[di].get(q) or 0) for di in doc_ids])
            score += (self.idf.get(q) or 0) * (q_freq * (self.k1 + 1) /
                                               (q_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)))
        return score.tolist()
```

get_scores와 get_batch_score, 그리고 내부 메소드로 IDF 계산법이 있음을 알 수 있다.  

rank_bm25의 get_scores 함수를 이용하여 결과에 Score를 추가한다. 

In [51]:
class BM25RetrieverAdvanced(BM25Retriever):

    def get_relevant_documents_with_score(self, query):
        # Implement the method to return documents with relevance scores
        results = self.invoke(query)
        scores = self.vectorizer.get_scores(query)
        return [(doc, score) for doc, score in zip(results, scores)]

In [52]:
bm25_retriever_advanced = BM25RetrieverAdvanced.from_documents(split_documents)


In [53]:
bm25_retriever_advanced.get_relevant_documents_with_score("Explain S&P 500 ETF")

[(Document(metadata={'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}, page_content='SPDR S&P 500 \nETF Trust (SPY) \nDelivering Unrivaled \nLiquidity to Investors'),
  3.2389831387529164),
 (Document(metadata={'document_id': '036658bb-b498-4d1d-8078-0a7aab520669', 'chunk_id': 0}, page_content='IVV\niShares Core S&P 500 ETF\nFact Sheet as of 31-Dec-2024\nThe iShares Core S&P 500 ETF seeks to track the investment results of an index \ncomposed of large-capitalization U.S. equities.\nWHY IVV?\n1 Exposure to large, established U.S. companies\n2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks\n3 Use at the core of your portfolio to seek long-term growth\nGROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION\nFund  Benchmark  \nThe Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes \nreinvestment of dividends and capital gains. Fund expenses, including management fees and \nother expenses were deducted.\nPERFORMANCE\n1 Year 3

정상적으로 relevant documents와 scores를 반환함을 확인했으니 rank_bm25의 get_top_n을 get_top_n_with_score로 추가해서 다시 수정한다.  

In [54]:
from rank_bm25 import BM25Okapi
import numpy as np

class BM25OkapaiModified(BM25Okapi):
    def __init__(self, corpus, tokenizer=None, k1=1.5, b=0.75, epsilon=0.25):
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon
        super().__init__(corpus, tokenizer)

    def get_top_n_with_score(self, query, documents, n=5):
        assert self.corpus_size == len(documents), "The documents given don't match the index corpus!"

        scores = self.get_scores(query)
        top_n = np.argsort(scores)[::-1][:n]
        return [(documents[i], scores[i]) for i in top_n]

In [55]:
bm25_instance = BM25OkapaiModified([doc.page_content for doc in split_documents])
dir(bm25_instance)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_calc_idf',
 '_initialize',
 '_tokenize_corpus',
 'average_idf',
 'avgdl',
 'b',
 'corpus_size',
 'doc_freqs',
 'doc_len',
 'epsilon',
 'get_batch_scores',
 'get_scores',
 'get_top_n',
 'get_top_n_with_score',
 'idf',
 'k1',
 'tokenizer']

In [63]:
from typing import Any, Callable, Dict, Iterable, List, Optional
from langchain_core.callbacks import CallbackManagerForRetrieverRun

def default_preprocessing_func(text: str) -> List[str]:
    return text.split()

class BM25RetrieverModified(BM25Retriever):

    @classmethod
    def from_texts(
        cls,
        texts: Iterable[str],
        metadatas: Optional[Iterable[dict]] = None,
        ids: Optional[Iterable[str]] = None,
        bm25_params: Optional[Dict[str, Any]] = None,
        preprocess_func: Callable[[str], List[str]] = default_preprocessing_func,
        **kwargs: Any,
    ) -> BM25Retriever:
        """
        Create a BM25Retriever from a list of texts.
        Args:
            texts: A list of texts to vectorize.
            metadatas: A list of metadata dicts to associate with each text.
            ids: A list of ids to associate with each text.
            bm25_params: Parameters to pass to the BM25 vectorizer.
            preprocess_func: A function to preprocess each text before vectorization.
            **kwargs: Any other arguments to pass to the retriever.

        Returns:
            A BM25Retriever instance.
        """
        try:
            from rank_bm25 import BM25Okapi
        except ImportError:
            raise ImportError(
                "Could not import rank_bm25, please install with `pip install "
                "rank_bm25`."
            )

        texts_processed = [preprocess_func(t) for t in texts]
        bm25_params = bm25_params or {}
        vectorizer = BM25OkapaiModified(texts_processed, **bm25_params)
        metadatas = metadatas or ({} for _ in texts)
        if ids:
            docs = [
                Document(page_content=t, metadata=m, id=i)
                for t, m, i in zip(texts, metadatas, ids)
            ]
        else:
            docs = [
                Document(page_content=t, metadata=m) for t, m in zip(texts, metadatas)
            ]
        return cls(
            vectorizer=vectorizer, docs=docs, preprocess_func=preprocess_func, **kwargs
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        processed_query = self.preprocess_func(query)
        return_docs = self.vectorizer.get_top_n_with_score(processed_query, self.docs, n=self.k)
        return return_docs

    def get_relevant_documents_with_score(self, query):
        # Implement the method to return documents with relevance scores
        results = self.invoke(query)
        scores = self.vectorizer.get_scores(query)
        return [(doc, score) for doc, score in zip(results, scores)]
    

In [64]:
# Set k as 10 to get top 10 relevant documents
bm25_retriever_modified = BM25RetrieverModified.from_documents(split_documents, k=10)

In [65]:
dir(bm25_retriever_modified)

['InputType',
 'OutputType',
 '__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__parameters__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__

get_relevant_documents_with_score와 invoke 메소드의 결과를 비교한다.    

In [66]:
bm25_retriever_modified.get_relevant_documents_with_score("Explain S&P 500 ETF")

[((Document(metadata={'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}, page_content='SPDR S&P 500 \nETF Trust (SPY) \nDelivering Unrivaled \nLiquidity to Investors'),
   2.4309272917303595),
  3.2389831387529164),
 ((Document(metadata={'document_id': '036658bb-b498-4d1d-8078-0a7aab520669', 'chunk_id': 0}, page_content='IVV\niShares Core S&P 500 ETF\nFact Sheet as of 31-Dec-2024\nThe iShares Core S&P 500 ETF seeks to track the investment results of an index \ncomposed of large-capitalization U.S. equities.\nWHY IVV?\n1 Exposure to large, established U.S. companies\n2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks\n3 Use at the core of your portfolio to seek long-term growth\nGROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION\nFund  Benchmark  \nThe Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes \nreinvestment of dividends and capital gains. Fund expenses, including management fees and \nother expenses were deduct

In [67]:
bm25_retriever_modified.invoke("Explain S&P 500 ETF")

[(Document(metadata={'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}, page_content='SPDR S&P 500 \nETF Trust (SPY) \nDelivering Unrivaled \nLiquidity to Investors'),
  2.4309272917303595),
 (Document(metadata={'document_id': '036658bb-b498-4d1d-8078-0a7aab520669', 'chunk_id': 0}, page_content='IVV\niShares Core S&P 500 ETF\nFact Sheet as of 31-Dec-2024\nThe iShares Core S&P 500 ETF seeks to track the investment results of an index \ncomposed of large-capitalization U.S. equities.\nWHY IVV?\n1 Exposure to large, established U.S. companies\n2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks\n3 Use at the core of your portfolio to seek long-term growth\nGROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION\nFund  Benchmark  \nThe Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes \nreinvestment of dividends and capital gains. Fund expenses, including management fees and \nother expenses were deducted.\nPERFORMANCE\n1 Year 3

invoke 메소드가 get_relevant_documents_with_score의 결과값을 정상적으로 반환함을 알 수 있다.  
또한 BM 25의 경우 score가 특정 구간으로 normalized 된 값이 아니다.  

### Dense Retrieve  

In langchain vectorstore, relevance score means socine similarity, inner_product, L2 distance    

In FAISS, similarity of two documents is defined as L2 distance.  

https://python.langchain.com/api_reference/_modules/langchain_community/vectorstores/faiss.html#FAISS.similarity_search_with_score  

https://python.langchain.com/api_reference/_modules/langchain_community/vectorstores/faiss.html#FAISS.similarity_search  

https://python.langchain.com/api_reference/_modules/langchain_community/vectorstores/utils.html#maximal_marginal_relevance

In [68]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# Embedding Model 생성

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Vector, L2 distance based Index  
index = faiss.IndexFlatL2(len(embeddings_model.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# FAISS 인덱스 생성
vector_store.add_documents(documents = split_documents)

['977af7ed-f8bb-4f03-8593-a01b33766eaf',
 '6ea6538d-0613-474f-b023-f52a563ade9f',
 'b817ec82-1680-41cb-9fac-05636c459060',
 '00fb778f-402e-474d-87cb-996ec8b20b8d',
 'd933e2cf-e5a0-4a4e-9da2-68ad7fbbc88c',
 '83938321-5354-4bf6-abf9-940f7c650129',
 '299f7bf6-2783-48d3-a858-66b927498291',
 'fcc85026-c7b6-47b7-a1cb-f8566dbb9b29',
 '7c8ceb26-2648-44a9-beaa-e1234a7d4941',
 '1597eefd-8127-4b56-90e1-84edb3d864e8',
 'f849556a-e606-4e9c-aff3-a66a5bcd6f1f',
 'f2b5b271-5152-4a2b-8e6d-01c09ec3e279',
 'b999d94a-93e8-4c62-80e1-4ba0ebc66247',
 '8de90fb5-f81c-43a4-a711-28d6865faa09',
 '3a3d46c3-55ec-479e-b982-964480773d2e',
 '01ef718d-655f-4db9-a695-33d4fe1b7ea1',
 '90e3ca30-ee24-42d5-916a-96a2eef46593',
 'a9a9ae33-7bf5-4e9c-8f51-992c90da05a5',
 '5f298174-063f-47df-9888-3f073a49bcf1',
 'f4b757c5-6c77-4d49-a33a-fa34100b17f8',
 'd09a20ab-5f07-485e-bcfb-2b09bc1cae97',
 'db202893-4f47-49df-ae48-cb3d2fa982b6',
 '8ebf2c36-40be-462d-ae64-688b62109909',
 '9430e19d-09d1-444d-8168-e17a1577b309',
 '965822e3-1c55-

In [69]:
query = 'Explain S&P 500 ETF'

results = vector_store.similarity_search_with_score(
    query, k=3
)

for res, score in results:
    print(f"* [SIM={score:3f}] Doc ID:{res.id} \n{res.page_content} [{res.metadata}]")

* [SIM=0.552862] Doc ID:00fb778f-402e-474d-87cb-996ec8b20b8d 
1
SPDR® S&P 500® 
ETF Trust
SPY
 
Fact Sheet
Equity
 
As of 12/31/2024
Key Features
 • The SPDR® S&P 500® ETF Trust seeks to provide investment 
results that, before expenses, correspond generally to 
the price and yield performance of the S&P 500® Index 
(the “Index”)
 • The S&P 500 Index is a diversified large cap U.S. index that 
holds companies across all eleven GICS sectors
 • Launched in January 1993, SPY was the very first exchange 
traded fund listed in the United States
About This Benchmark
The S&P 500® Index is designed to measure the performance of 
the large-cap segment of the US equity market. It is float-adjusted 
market capitalization weighted. 
Fund Information
Inception Date 01/22/1993
CUSIP 78462F103
T otal Return (As of 12/31/2024)
 NAV
(%)
Market Value
(%)
Index
(%)
Cumulative    
QTD 2.38 2.43 2.41
YTD 24.87 24.86 25.02
Annualized    
1 Year 24.87 24.86 25.02
3 Year 8.81 8.81 8.94
5 Year 14.38 14.40 14.5

In FAISS, the similarity is L2 distance and therefore the range of similarity lies in [0, 1].  

## Ensemble Retriever  

Ensemble Retriever in langchain offers RRF ans weighted RRF.  

https://github.com/langchain-ai/langchain/blob/master/libs/langchain/langchain/retrievers/ensemble.py  

**Ensemble Methods**  

1. Wegithed Average or Weighted Sum  
2. Hard Voting  
3. Borda Count  
4. RRF (Reciprocal Rank Fusion)
5. Weighted RRF   


### Weighted Average or Sum  

BM 25의 경우 score의 normalization이 필요하다.  

Convert the BM 25 score to a value in the range [0, 1].  

In [70]:
query = "Explain S&P 500 ETF"
sparse_results = bm25_retriever_modified.invoke(query)
dense_results = vector_store.similarity_search_with_score(query, k=10)

In [71]:
def convert_sparse_score(sparse_results):
    scores = [score for _, score in sparse_results]
    max_score = max(scores)
    min_score = min(scores)
    converted_scores = [(score - min_score) / (max_score - min_score) for score in scores]
    converted_sparse_results = []
    for i, (doc, _) in enumerate(sparse_results):
        converted_sparse_results.append((doc, converted_scores[i]))
    return converted_sparse_results

In [72]:
converted_sparse_results = convert_sparse_score(sparse_results)
converted_sparse_results

[(Document(metadata={'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}, page_content='SPDR S&P 500 \nETF Trust (SPY) \nDelivering Unrivaled \nLiquidity to Investors'),
  1.0),
 (Document(metadata={'document_id': '036658bb-b498-4d1d-8078-0a7aab520669', 'chunk_id': 0}, page_content='IVV\niShares Core S&P 500 ETF\nFact Sheet as of 31-Dec-2024\nThe iShares Core S&P 500 ETF seeks to track the investment results of an index \ncomposed of large-capitalization U.S. equities.\nWHY IVV?\n1 Exposure to large, established U.S. companies\n2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks\n3 Use at the core of your portfolio to seek long-term growth\nGROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION\nFund  Benchmark  \nThe Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes \nreinvestment of dividends and capital gains. Fund expenses, including management fees and \nother expenses were deducted.\nPERFORMANCE\n1 Year 3 Year 5 Year 10

In [73]:
dense_results

[(Document(id='00fb778f-402e-474d-87cb-996ec8b20b8d', metadata={'document_id': '8080929a-1cc7-4c63-890c-ffb72027edc6', 'chunk_id': 0}, page_content='1\nSPDR® S&P 500® \nETF Trust\nSPY\n \nFact Sheet\nEquity\n \nAs of 12/31/2024\nKey Features\n • The SPDR® S&P 500® ETF Trust seeks to provide investment \nresults that, before expenses, correspond generally to \nthe price and yield performance of the S&P 500® Index \n(the “Index”)\n • The S&P 500 Index is a diversified large cap U.S. index that \nholds companies across all eleven GICS sectors\n • Launched in January 1993, SPY was the very first exchange \ntraded fund listed in the United States\nAbout This Benchmark\nThe S&P 500® Index is designed to measure the performance of \nthe large-cap segment of the US equity market. It is float-adjusted \nmarket capitalization weighted. \nFund Information\nInception Date 01/22/1993\nCUSIP 78462F103\nT otal Return (As of 12/31/2024)\n NAV\n(%)\nMarket Value\n(%)\nIndex\n(%)\nCumulative    \nQTD 2.

Similarity가 ascending order다. 높은게 위로 오도록 다시 정렬한다.  

In [74]:
dense_results = dense_results[::-1]
dense_results

[(Document(id='93a3b156-cb61-484a-ab9e-4cd86657a692', metadata={'document_id': 'cc8274bb-2c23-4802-bd80-de00c8862c13', 'chunk_id': 0}, page_content='Vanguard®\nVanguard Total Stock Market ETF   |  VTI As of December 31, 2024 \nInvestment approach •Seeks to track the performance of the CRSP US Total Market Index. •Large-, mid-, and small-cap equity diversified across growth and value styles. •Employs apassively managed, index-sampling strategy. •The fund remains fully invested. •Low expenses minimize net tracking error. About the benchmark •The CRSP US Total Market Index represents approximately 100% of investable companies in the U.S. equity market. •The index is designed to accurately represent the U.S. equity market and deliver low turnover. Performance history Total returns  2 for period ended December 31, 2024    \nVTI (Inception 2001-05-24)  Quarter Year to date  1 year 3years 5years 10 years Since inception Net asset value (NAV) return 3 2.62% 23.75% 23.75% 7.88% 13.80% 12.50% 8.

### Hard Voting  

In [75]:
for result, score in converted_sparse_results:
    print(result.metadata)
    print(result.page_content)
    print(score)
    break

{'document_id': '81f9053a-d441-4fdc-817b-f136662ef34e', 'chunk_id': 0}
SPDR S&P 500 
ETF Trust (SPY) 
Delivering Unrivaled 
Liquidity to Investors
1.0


In [76]:
for result, score in dense_results:
    print(result.id)
    print(result.metadata)
    print(result.page_content)
    print(score)
    break

93a3b156-cb61-484a-ab9e-4cd86657a692
{'document_id': 'cc8274bb-2c23-4802-bd80-de00c8862c13', 'chunk_id': 0}
Vanguard®
Vanguard Total Stock Market ETF   |  VTI As of December 31, 2024 
Investment approach •Seeks to track the performance of the CRSP US Total Market Index. •Large-, mid-, and small-cap equity diversified across growth and value styles. •Employs apassively managed, index-sampling strategy. •The fund remains fully invested. •Low expenses minimize net tracking error. About the benchmark •The CRSP US Total Market Index represents approximately 100% of investable companies in the U.S. equity market. •The index is designed to accurately represent the U.S. equity market and deliver low turnover. Performance history Total returns  2 for period ended December 31, 2024    
VTI (Inception 2001-05-24)  Quarter Year to date  1 year 3years 5years 10 years Since inception Net asset value (NAV) return 3 2.62% 23.75% 23.75% 7.88% 13.80% 12.50% 8.89% Market price return 4 2.68 23.71 23.71 7

In [77]:
sparse_document_ids = set([doc.metadata['document_id'] for doc, _ in converted_sparse_results])
dense_document_ids = set([doc.metadata['document_id'] for doc, _ in dense_results])
common_document_ids = sparse_document_ids.intersection(dense_document_ids)
common_document_ids

{'036658bb-b498-4d1d-8078-0a7aab520669',
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f',
 '8080929a-1cc7-4c63-890c-ffb72027edc6',
 '81f9053a-d441-4fdc-817b-f136662ef34e',
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0',
 'afa1f693-7e09-4109-a53b-fb098a3efa28',
 'd34f5126-dd3f-450a-a846-a82e943ef145',
 'dcc58dfb-f428-4948-ae1e-90143779ae17'}

In [78]:
common_documents = [doc for doc, _ in dense_results if doc.metadata['document_id'] in common_document_ids]
common_documents

[Document(id='fcc85026-c7b6-47b7-a1cb-f8566dbb9b29', metadata={'document_id': 'dcc58dfb-f428-4948-ae1e-90143779ae17', 'chunk_id': 0}, page_content='2025년01월31일\n \n \n \n※삼성자산운용주식회사는 이 투자신탁의 투자대상 자산의 종류 및 위험도를\n감안하여 2등급으로 분류하였습니다. 펀드의 위험 등급은 운용실적, 시장 상황\n등에 따라 변경될 수 있다는 점을 유의하여 투자판단을 하시기 바랍니다.\n2025년 01월 31일 기준\nKodex 미국S&P500\n \n (379800)\nS&P500 Total Return Index(KRW)를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운\n용하여 투자대상 자산의 가치상승에 따른 수익을 추구합니다.\n- 미국 증권시장 상장 대형주 500여개 종목에 투자하는 미국 대표지수 ETF\n- 배당 재투자를 통한 장기 복리 효과 기대\n기본정보\nETF명 Kodex 미국S&P500\n거래소코드 379800\n기초지수 S&P 500 Total Return Index\nETF순자산총액       37,917.08억원\n1주당 NAV       20,157.94원\n총 보수 연 0.0099%\n(지정판매 0.001%, 집합투자 0.0009%, 신탁 0.005%, 일반사무 0.003%)\n상장일 2021.04.09\n분배금 미지급(발생 시 재투자)\n운용회사 삼성자산운용\n사무수탁회사 신한펀드파트너스\n수탁은행 한국씨티은행\n환매방법 유가증권 시장을 통한 매도, 지정참가회사를 통한 해지에 의한 환매\n설정단위 50,000주\n \n본 자료는 펀드의 단순 정보제공을 위해 작성된 것으로써, 본 펀드의 투자광고 및 투자권유를 위해 작성된 자료가 아닙니다. 따라서 본 자료는 삼성자산운용 홈페이지 게시 외에는 본 펀드에 가입하지 않은 고객에게 투자광고 또는 투자권유의\n

In [79]:
def intersection_items(retriver_results):
    common_document_ids = set([doc.metadata['document_id'] for doc, _ in retriver_results[0]])
    for results in retriver_results[1:]:
        document_ids = set([doc.metadata['document_id'] for doc, _ in results])
        common_document_ids = common_document_ids.intersection(document_ids)
    return common_documents

In [80]:
gathered_intersection = intersection_items([converted_sparse_results, dense_results])
gathered_intersection

[Document(id='fcc85026-c7b6-47b7-a1cb-f8566dbb9b29', metadata={'document_id': 'dcc58dfb-f428-4948-ae1e-90143779ae17', 'chunk_id': 0}, page_content='2025년01월31일\n \n \n \n※삼성자산운용주식회사는 이 투자신탁의 투자대상 자산의 종류 및 위험도를\n감안하여 2등급으로 분류하였습니다. 펀드의 위험 등급은 운용실적, 시장 상황\n등에 따라 변경될 수 있다는 점을 유의하여 투자판단을 하시기 바랍니다.\n2025년 01월 31일 기준\nKodex 미국S&P500\n \n (379800)\nS&P500 Total Return Index(KRW)를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운\n용하여 투자대상 자산의 가치상승에 따른 수익을 추구합니다.\n- 미국 증권시장 상장 대형주 500여개 종목에 투자하는 미국 대표지수 ETF\n- 배당 재투자를 통한 장기 복리 효과 기대\n기본정보\nETF명 Kodex 미국S&P500\n거래소코드 379800\n기초지수 S&P 500 Total Return Index\nETF순자산총액       37,917.08억원\n1주당 NAV       20,157.94원\n총 보수 연 0.0099%\n(지정판매 0.001%, 집합투자 0.0009%, 신탁 0.005%, 일반사무 0.003%)\n상장일 2021.04.09\n분배금 미지급(발생 시 재투자)\n운용회사 삼성자산운용\n사무수탁회사 신한펀드파트너스\n수탁은행 한국씨티은행\n환매방법 유가증권 시장을 통한 매도, 지정참가회사를 통한 해지에 의한 환매\n설정단위 50,000주\n \n본 자료는 펀드의 단순 정보제공을 위해 작성된 것으로써, 본 펀드의 투자광고 및 투자권유를 위해 작성된 자료가 아닙니다. 따라서 본 자료는 삼성자산운용 홈페이지 게시 외에는 본 펀드에 가입하지 않은 고객에게 투자광고 또는 투자권유의\n

In [81]:
def hard_voting(retriver_results):
    voting_dict = {doc.metadata['document_id']: 1 for doc, _ in retriver_results[0]}
    for results in retriver_results[1:]:
        for doc, score in results:
            if doc.metadata['document_id'] in voting_dict:
                voting_dict[doc.metadata['document_id']] += 1
            else:
                voting_dict[doc.metadata['document_id']] = 1
    return voting_dict

In [82]:
gathered_hard_voting = hard_voting([converted_sparse_results, dense_results])
gathered_hard_voting

{'81f9053a-d441-4fdc-817b-f136662ef34e': 2,
 '036658bb-b498-4d1d-8078-0a7aab520669': 2,
 'd34f5126-dd3f-450a-a846-a82e943ef145': 2,
 '30593878-327e-4d80-9187-e6be901eaea1': 1,
 'afa1f693-7e09-4109-a53b-fb098a3efa28': 2,
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0': 2,
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f': 2,
 'dcc58dfb-f428-4948-ae1e-90143779ae17': 2,
 '8080929a-1cc7-4c63-890c-ffb72027edc6': 2,
 '519e4fc3-5fed-479b-878c-f8df59b1e945': 1,
 'cc8274bb-2c23-4802-bd80-de00c8862c13': 1,
 'afa3c9d9-1b36-4b45-8695-8d33e24f6607': 1}

### Borda Count  

In [83]:
def get_borda_point(results):
    points = {}
    k = len(results)
    for i, (doc, score) in enumerate(results):
        # get methods returns None if key is NOT found
        points[doc.metadata['document_id']] = points.get(doc.metadata['document_id'], 0) + (k - i - 1)
    return points

In [84]:
sparse_borda_points = get_borda_point(converted_sparse_results)
sparse_borda_points

{'81f9053a-d441-4fdc-817b-f136662ef34e': 9,
 '036658bb-b498-4d1d-8078-0a7aab520669': 8,
 'd34f5126-dd3f-450a-a846-a82e943ef145': 7,
 '30593878-327e-4d80-9187-e6be901eaea1': 6,
 'afa1f693-7e09-4109-a53b-fb098a3efa28': 5,
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0': 4,
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f': 3,
 'dcc58dfb-f428-4948-ae1e-90143779ae17': 2,
 '8080929a-1cc7-4c63-890c-ffb72027edc6': 1,
 '519e4fc3-5fed-479b-878c-f8df59b1e945': 0}

In [85]:
dense_borda_points = get_borda_point(dense_results)
dense_borda_points

{'cc8274bb-2c23-4802-bd80-de00c8862c13': 9,
 'dcc58dfb-f428-4948-ae1e-90143779ae17': 8,
 'd34f5126-dd3f-450a-a846-a82e943ef145': 7,
 'afa3c9d9-1b36-4b45-8695-8d33e24f6607': 6,
 '036658bb-b498-4d1d-8078-0a7aab520669': 5,
 '81f9053a-d441-4fdc-817b-f136662ef34e': 4,
 'afa1f693-7e09-4109-a53b-fb098a3efa28': 3,
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0': 2,
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f': 1,
 '8080929a-1cc7-4c63-890c-ffb72027edc6': 0}

In [86]:
def gather_borda_points(borda_points_list):
    gathered_borda_points = borda_points_list[0]
    for results in borda_points_list[1:]:
        for doc_id, point in results.items():
            gathered_borda_points[doc_id] = gathered_borda_points.get(doc_id, 0) + point
        
    return gathered_borda_points

In [87]:
gathered_borda_points = gather_borda_points([sparse_borda_points, dense_borda_points])
gathered_borda_points

{'81f9053a-d441-4fdc-817b-f136662ef34e': 13,
 '036658bb-b498-4d1d-8078-0a7aab520669': 13,
 'd34f5126-dd3f-450a-a846-a82e943ef145': 14,
 '30593878-327e-4d80-9187-e6be901eaea1': 6,
 'afa1f693-7e09-4109-a53b-fb098a3efa28': 8,
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0': 6,
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f': 4,
 'dcc58dfb-f428-4948-ae1e-90143779ae17': 10,
 '8080929a-1cc7-4c63-890c-ffb72027edc6': 1,
 '519e4fc3-5fed-479b-878c-f8df59b1e945': 0,
 'cc8274bb-2c23-4802-bd80-de00c8862c13': 9,
 'afa3c9d9-1b36-4b45-8695-8d33e24f6607': 6}

In [88]:
def convert_borda_point_to_rank(borda_points):
    #sorted_borda_ranks = sorted(borda_points.items(), key=lambda x: x[1], reverse=True)
    # 내림차순 정렬 인덱스
    sorted_borda_ranks = [i for _, i in sorted(zip(borda_points, range(len(borda_points))), reverse=True)]
    return sorted_borda_ranks

In [89]:
gathered_borda_ranks = convert_borda_point_to_rank(gathered_borda_points)
gathered_borda_ranks

[7, 2, 10, 11, 4, 5, 0, 8, 6, 9, 3, 1]

In [90]:
sorted(zip(gathered_borda_points, range(len(gathered_borda_points))), reverse=True)

[('dcc58dfb-f428-4948-ae1e-90143779ae17', 7),
 ('d34f5126-dd3f-450a-a846-a82e943ef145', 2),
 ('cc8274bb-2c23-4802-bd80-de00c8862c13', 10),
 ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 11),
 ('afa1f693-7e09-4109-a53b-fb098a3efa28', 4),
 ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 5),
 ('81f9053a-d441-4fdc-817b-f136662ef34e', 0),
 ('8080929a-1cc7-4c63-890c-ffb72027edc6', 8),
 ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 6),
 ('519e4fc3-5fed-479b-878c-f8df59b1e945', 9),
 ('30593878-327e-4d80-9187-e6be901eaea1', 3),
 ('036658bb-b498-4d1d-8078-0a7aab520669', 1)]

In [91]:
import pandas as pd
gathered_borda_points_df = pd.DataFrame(gathered_borda_points.items(), columns=['document_id', 'borda_point'])
gathered_borda_points_df

,document_id,borda_point
0,81f9053a-d441-4fdc-817b-f136662ef34e,13
1,036658bb-b498-4d1d-8078-0a7aab520669,13
2,d34f5126-dd3f-450a-a846-a82e943ef145,14
3,30593878-327e-4d80-9187-e6be901eaea1,6
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10
8,8080929a-1cc7-4c63-890c-ffb72027edc6,1
9,519e4fc3-5fed-479b-878c-f8df59b1e945,0


In [92]:
gathered_borda_points_df.sort_values(by='borda_point', ascending=False)


,document_id,borda_point
2,d34f5126-dd3f-450a-a846-a82e943ef145,14
0,81f9053a-d441-4fdc-817b-f136662ef34e,13
1,036658bb-b498-4d1d-8078-0a7aab520669,13
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8
3,30593878-327e-4d80-9187-e6be901eaea1,6
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4


In [93]:
def get_borda_ranks(borda_points):
    df = pd.DataFrame(borda_points.items(), columns=['document_id', 'borda_point'])
    # Sort by Rank in Descending order
    df = df.sort_values(by='borda_point', ascending=False)
    df['borda_rank'] = range(1, len(df) + 1)
    sorted_borda_ranks = df
    #sorted_borda_ranks = df.to_dict()
    return sorted_borda_ranks

In [94]:
gathered_borda_ranks = get_borda_ranks(gathered_borda_points)
gathered_borda_ranks

,document_id,borda_point,borda_rank
2,d34f5126-dd3f-450a-a846-a82e943ef145,14,1
0,81f9053a-d441-4fdc-817b-f136662ef34e,13,2
1,036658bb-b498-4d1d-8078-0a7aab520669,13,3
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10,4
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9,5
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8,6
3,30593878-327e-4d80-9187-e6be901eaea1,6,7
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6,8
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6,9
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4,10


In [95]:
k = 10
gathered_top_document_ids = gathered_borda_ranks['document_id'].tolist()[:k]
gathered_top_document_ids


['d34f5126-dd3f-450a-a846-a82e943ef145',
 '81f9053a-d441-4fdc-817b-f136662ef34e',
 '036658bb-b498-4d1d-8078-0a7aab520669',
 'dcc58dfb-f428-4948-ae1e-90143779ae17',
 'cc8274bb-2c23-4802-bd80-de00c8862c13',
 'afa1f693-7e09-4109-a53b-fb098a3efa28',
 '30593878-327e-4d80-9187-e6be901eaea1',
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0',
 'afa3c9d9-1b36-4b45-8695-8d33e24f6607',
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f']

In [96]:
def convert_borda_point_to_rank(borda_points):
    #sorted_borda_ranks = sorted(borda_points.items(), key=lambda x: x[1], reverse=True)
    # 내림차순 정렬 인덱스
    sorted_borda_ranks = [i for _, i in sorted(zip(borda_points, range(len(borda_points))), reverse=True)]
    return sorted_borda_ranks

def gather_borda_points_ranks(borda_points_list):
    gathered_borda_points = borda_points_list[0]
    for results in borda_points_list[1:]:
        for doc_id, point in results.items():
            gathered_borda_points[doc_id] = gathered_borda_points.get(doc_id, 0) + point
    gathered_borda_results = [zip(gathered_borda_points.keys(), i) for i in sorted(gathered_borda_points.items(), key=lambda s: gathered_borda_points.items[0], reverse=True)]
        
    return gathered_borda_points

In [97]:
gathered_borda_points

{'81f9053a-d441-4fdc-817b-f136662ef34e': 13,
 '036658bb-b498-4d1d-8078-0a7aab520669': 13,
 'd34f5126-dd3f-450a-a846-a82e943ef145': 14,
 '30593878-327e-4d80-9187-e6be901eaea1': 6,
 'afa1f693-7e09-4109-a53b-fb098a3efa28': 8,
 '8d311550-6ba0-49f2-b8a4-fcd437d20fa0': 6,
 '7eb0e53a-ee34-472b-b275-a0cb50a03d9f': 4,
 'dcc58dfb-f428-4948-ae1e-90143779ae17': 10,
 '8080929a-1cc7-4c63-890c-ffb72027edc6': 1,
 '519e4fc3-5fed-479b-878c-f8df59b1e945': 0,
 'cc8274bb-2c23-4802-bd80-de00c8862c13': 9,
 'afa3c9d9-1b36-4b45-8695-8d33e24f6607': 6}

In [98]:
sorted(gathered_borda_points.items(), key=lambda x: x[1], reverse=True)

[('d34f5126-dd3f-450a-a846-a82e943ef145', 14),
 ('81f9053a-d441-4fdc-817b-f136662ef34e', 13),
 ('036658bb-b498-4d1d-8078-0a7aab520669', 13),
 ('dcc58dfb-f428-4948-ae1e-90143779ae17', 10),
 ('cc8274bb-2c23-4802-bd80-de00c8862c13', 9),
 ('afa1f693-7e09-4109-a53b-fb098a3efa28', 8),
 ('30593878-327e-4d80-9187-e6be901eaea1', 6),
 ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 6),
 ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 6),
 ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 4),
 ('8080929a-1cc7-4c63-890c-ffb72027edc6', 1),
 ('519e4fc3-5fed-479b-878c-f8df59b1e945', 0)]

In [99]:
gathered_borda_points.items()

dict_items([('81f9053a-d441-4fdc-817b-f136662ef34e', 13), ('036658bb-b498-4d1d-8078-0a7aab520669', 13), ('d34f5126-dd3f-450a-a846-a82e943ef145', 14), ('30593878-327e-4d80-9187-e6be901eaea1', 6), ('afa1f693-7e09-4109-a53b-fb098a3efa28', 8), ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 6), ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 4), ('dcc58dfb-f428-4948-ae1e-90143779ae17', 10), ('8080929a-1cc7-4c63-890c-ffb72027edc6', 1), ('519e4fc3-5fed-479b-878c-f8df59b1e945', 0), ('cc8274bb-2c23-4802-bd80-de00c8862c13', 9), ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 6)])

In [100]:
[(item[0], item[1], idx + 1) for idx, item in enumerate(sorted(gathered_borda_points.items(), key=lambda x: x[1], reverse=True))]

[('d34f5126-dd3f-450a-a846-a82e943ef145', 14, 1),
 ('81f9053a-d441-4fdc-817b-f136662ef34e', 13, 2),
 ('036658bb-b498-4d1d-8078-0a7aab520669', 13, 3),
 ('dcc58dfb-f428-4948-ae1e-90143779ae17', 10, 4),
 ('cc8274bb-2c23-4802-bd80-de00c8862c13', 9, 5),
 ('afa1f693-7e09-4109-a53b-fb098a3efa28', 8, 6),
 ('30593878-327e-4d80-9187-e6be901eaea1', 6, 7),
 ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 6, 8),
 ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 6, 9),
 ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 4, 10),
 ('8080929a-1cc7-4c63-890c-ffb72027edc6', 1, 11),
 ('519e4fc3-5fed-479b-878c-f8df59b1e945', 0, 12)]

위 Borda count의 경우 동점이 생긴다. 따라서 동점을 처리할 방법이 필요하다.  

### 동점 처리 Tie Score Processing  

1. 동일 순위 부여  
2. 평균 순위 부여  
3. 동점자 동순위 부여  

In [101]:
def calculate_dense_ranking(scores):
    """
    점수 목록을 입력받아 동일 순위 부여 방식으로 순위를 매기는 함수

    Args:
        scores: 점수 목록 (리스트)

    Returns:
        순위가 매겨진 점수 목록 (리스트)
    """

    # 점수를 내림차순으로 정렬
    sorted_scores = sorted(scores, reverse=True)

    # 순위와 순위 인덱스 초기화
    rank = 1
    rank_index = 0

    # 결과를 저장할 리스트
    ranked_scores = []

    while rank_index < len(sorted_scores):
        # 현재 점수
        current_score = sorted_scores[rank_index]

        # 동점자 수
        same_score_count = 1

        # 동점자 수 계산
        for i in range(rank_index + 1, len(sorted_scores)):
            if sorted_scores[i] == current_score:
                same_score_count += 1
            else:
                break

        # 동점자에게 같은 순위 부여
        for _ in range(same_score_count):
            ranked_scores.append((current_score, rank))

        # 다음 순위 갱신 (동점자 수만큼 증가)
        rank += same_score_count

        # 다음 순위 인덱스 갱신 
        rank_index += same_score_count

    return ranked_scores


# 예시 점수 목록
scores = [85, 92, 78, 92, 85, 95, 80]

# 순위 매기기
ranked_scores = calculate_dense_ranking(scores)

# 결과 출력
for score, rank in ranked_scores:
    print(f"점수: {score}, 순위: {rank}")

점수: 95, 순위: 1
점수: 92, 순위: 2
점수: 92, 순위: 2
점수: 85, 순위: 4
점수: 85, 순위: 4
점수: 80, 순위: 6
점수: 78, 순위: 7


In [102]:
def calculate_fractional_ranking(scores):
    """
    점수 목록을 입력받아 평균 순위 부여 방식으로 순위를 매기는 함수

    Args:
        scores: 점수 목록 (리스트)

    Returns:
        순위가 매겨진 점수 목록 (리스트)
    """

    # 점수를 내림차순으로 정렬
    sorted_scores = sorted(scores, reverse=True)

    # 순위와 순위 인덱스 초기화
    rank = 1
    rank_index = 0

    # 결과를 저장할 리스트
    ranked_scores = []

    while rank_index < len(sorted_scores):
        # 현재 점수
        current_score = sorted_scores[rank_index]

        # 동점자 수
        same_score_count = 1

        # 동점자 수 계산
        for i in range(rank_index + 1, len(sorted_scores)):
            if sorted_scores[i] == current_score:
                same_score_count += 1
            else:
                break

        # 동점자 평균 순위 계산
        average_rank = sum(range(rank, rank + same_score_count)) / same_score_count

        # 동점자에게 평균 순위 부여
        for _ in range(same_score_count):
            ranked_scores.append((current_score, average_rank))

        # 다음 순위 갱신 (다음 순위는 항상 동점자 수만큼 증가)
        rank += same_score_count

        # 다음 순위 인덱스 갱신
        rank_index += same_score_count

    return ranked_scores


# 예시 점수 목록
scores = [85, 92, 78, 92, 85, 95, 80]

# 순위 매기기
ranked_scores = calculate_fractional_ranking(scores)

# 결과 출력
for score, rank in ranked_scores:
    print(f"점수: {score}, 순위: {rank}")

점수: 95, 순위: 1.0
점수: 92, 순위: 2.5
점수: 92, 순위: 2.5
점수: 85, 순위: 4.5
점수: 85, 순위: 4.5
점수: 80, 순위: 6.0
점수: 78, 순위: 7.0


In [103]:
def calculate_ordinal_ranking(scores):
    """
    점수 목록을 입력받아 순위를 계산하는 함수

    Args:
        scores: 점수 목록 (리스트)

    Returns:
        점수와 순위가 매겨진 목록 (리스트)
    """

    # 점수를 내림차순으로 정렬
    sorted_scores = sorted(scores, reverse=True)

    # 순위와 순위 인덱스 초기화
    rank = 1
    rank_index = 0

    # 결과를 저장할 리스트
    ranked_scores = []

    while rank_index < len(sorted_scores):
        # 현재 점수
        current_score = sorted_scores[rank_index]

        # 동점자 수
        same_score_count = 1

        # 동점자 수 계산
        for i in range(rank_index + 1, len(sorted_scores)):
            if sorted_scores[i] == current_score:
                same_score_count += 1
            else:
                break

        # 동점자에게 같은 순위 부여
        for _ in range(same_score_count):
            ranked_scores.append((current_score, rank))

        # 다음 순위 갱신 (동점자 수가 아닌 1만큼만 증가)
        rank += 1

        # 다음 순위 인덱스 갱신
        rank_index += same_score_count

    return ranked_scores


# 예시 점수 목록
scores = [100, 92, 92, 90, 85, 85, 85, 80]

# 순위 계산
ranked_scores = calculate_ordinal_ranking(scores)

# 결과 출력
for score, rank in ranked_scores:
    print(f"점수: {score}, 순위: {rank}")

점수: 100, 순위: 1
점수: 92, 순위: 2
점수: 92, 순위: 2
점수: 90, 순위: 3
점수: 85, 순위: 4
점수: 85, 순위: 4
점수: 85, 순위: 4
점수: 80, 순위: 5


In [104]:
def get_borda_ranks(borda_points, tie_breaker=None):
    # Convert Borda Points to Ranks
    df = pd.DataFrame(borda_points.items(), columns=['document_id', 'borda_point'])
    # Sort by Rank in Descending order
    df = df.sort_values(by='borda_point', ascending=False)
    assert tie_breaker in ['dense', 'fractional', 'ordinal', None], "Invalid tie_breaker"
    if tie_breaker == 'dense':
        ranked_scores = calculate_dense_ranking(df['borda_point'].to_numpy())
        ranks = [rank for _, rank in ranked_scores]
        df['borda_rank'] = ranks
    elif tie_breaker == 'fractional':
        ranked_scores = calculate_fractional_ranking(df['borda_point'].to_numpy())
        ranks = [rank for _, rank in ranked_scores]
        df['borda_rank'] = ranks
    elif tie_breaker == 'ordinal':
        ranked_scores = calculate_ordinal_ranking(df['borda_point'].to_numpy())
        ranks = [rank for _, rank in ranked_scores]
        df['borda_rank'] = ranks
    else:   
        df['borda_rank'] = range(1, len(df) + 1)
    sorted_borda_ranks = df
    #sorted_borda_ranks = df.to_dict()
    return sorted_borda_ranks

gathered_borda_ranks = get_borda_ranks(gathered_borda_points)
gathered_borda_ranks

,document_id,borda_point,borda_rank
2,d34f5126-dd3f-450a-a846-a82e943ef145,14,1
0,81f9053a-d441-4fdc-817b-f136662ef34e,13,2
1,036658bb-b498-4d1d-8078-0a7aab520669,13,3
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10,4
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9,5
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8,6
3,30593878-327e-4d80-9187-e6be901eaea1,6,7
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6,8
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6,9
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4,10


In [105]:
gathered_borda_ranks = get_borda_ranks(gathered_borda_points, tie_breaker='dense')
gathered_borda_ranks

,document_id,borda_point,borda_rank
2,d34f5126-dd3f-450a-a846-a82e943ef145,14,1
0,81f9053a-d441-4fdc-817b-f136662ef34e,13,2
1,036658bb-b498-4d1d-8078-0a7aab520669,13,2
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10,4
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9,5
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8,6
3,30593878-327e-4d80-9187-e6be901eaea1,6,7
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6,7
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6,7
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4,10


In [106]:
gathered_borda_ranks = get_borda_ranks(gathered_borda_points, tie_breaker='fractional')
gathered_borda_ranks

,document_id,borda_point,borda_rank
2,d34f5126-dd3f-450a-a846-a82e943ef145,14,1.0
0,81f9053a-d441-4fdc-817b-f136662ef34e,13,2.5
1,036658bb-b498-4d1d-8078-0a7aab520669,13,2.5
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10,4.0
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9,5.0
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8,6.0
3,30593878-327e-4d80-9187-e6be901eaea1,6,8.0
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6,8.0
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6,8.0
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4,10.0


In [107]:
gathered_borda_ranks = get_borda_ranks(gathered_borda_points, tie_breaker='ordinal')
gathered_borda_ranks

,document_id,borda_point,borda_rank
2,d34f5126-dd3f-450a-a846-a82e943ef145,14,1
0,81f9053a-d441-4fdc-817b-f136662ef34e,13,2
1,036658bb-b498-4d1d-8078-0a7aab520669,13,2
7,dcc58dfb-f428-4948-ae1e-90143779ae17,10,3
10,cc8274bb-2c23-4802-bd80-de00c8862c13,9,4
4,afa1f693-7e09-4109-a53b-fb098a3efa28,8,5
3,30593878-327e-4d80-9187-e6be901eaea1,6,6
5,8d311550-6ba0-49f2-b8a4-fcd437d20fa0,6,6
11,afa3c9d9-1b36-4b45-8695-8d33e24f6607,6,6
6,7eb0e53a-ee34-472b-b275-a0cb50a03d9f,4,7


### RRF (Reciprocal Rank Fusion)  

In [108]:
def reciprocal_rank_fusion(ranked_lists):
    """
    여러 개의 검색 결과 목록을 RRF를 이용하여 융합합니다.
    The Score is reciprocal of rank

    Args:
        ranked_lists: 검색 결과 목록들의 리스트. 각 목록은 (문서 ID, 순위) 튜플의 리스트입니다.

    Returns:
        융합된 검색 결과 목록. (문서 ID, 점수) 튜플의 리스트입니다.
    """

    document_scores = {}
    for ranked_list in ranked_lists:
        for i, (doc, _) in enumerate(ranked_list):
            doc_id = doc.metadata['document_id']
            if doc_id not in document_scores:
                document_scores[doc_id] = 0
            document_scores[doc_id] += 1 / (i + 1)  # 순위는 1부터 시작하므로 i + 1

    sorted_scores = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_scores

'''
# 예시 검색 결과 목록
ranked_lists = [
    [("A", 1), ("B", 2), ("C", 3)],
    [("B", 1), ("A", 2), ("D", 3)],
    [("A", 1), ("C", 2), ("D", 3)],
]

# RRF 융합
fused_results = reciprocal_rank_fusion(ranked_lists)

# 결과 출력
print(fused_results)
'''

'\n# 예시 검색 결과 목록\nranked_lists = [\n    [("A", 1), ("B", 2), ("C", 3)],\n    [("B", 1), ("A", 2), ("D", 3)],\n    [("A", 1), ("C", 2), ("D", 3)],\n]\n\n# RRF 융합\nfused_results = reciprocal_rank_fusion(ranked_lists)\n\n# 결과 출력\nprint(fused_results)\n'

In [109]:
ranked_lists = [converted_sparse_results, dense_results]

# RRF 융합
fused_results = reciprocal_rank_fusion(ranked_lists)

# 결과 출력
print(f"len of fused_results: {len(fused_results)}")
print(fused_results)

len of fused_results: 12
[('81f9053a-d441-4fdc-817b-f136662ef34e', 1.1666666666666667), ('cc8274bb-2c23-4802-bd80-de00c8862c13', 1.0), ('036658bb-b498-4d1d-8078-0a7aab520669', 0.7), ('d34f5126-dd3f-450a-a846-a82e943ef145', 0.6666666666666666), ('dcc58dfb-f428-4948-ae1e-90143779ae17', 0.625), ('afa1f693-7e09-4109-a53b-fb098a3efa28', 0.34285714285714286), ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 0.29166666666666663), ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 0.25396825396825395), ('30593878-327e-4d80-9187-e6be901eaea1', 0.25), ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 0.25), ('8080929a-1cc7-4c63-890c-ffb72027edc6', 0.2111111111111111), ('519e4fc3-5fed-479b-878c-f8df59b1e945', 0.1)]


### Weighted RRF  

In [110]:
def weighted_reciprocal_rank_fusion(ranked_lists, weights):
    """
    여러 개의 검색 결과 목록을 가중 RRF를 이용하여 융합합니다.

    Args:
        ranked_lists: 검색 결과 목록들의 리스트. 각 목록은 (문서 ID, 순위) 튜플의 리스트입니다.
        weights: 각 검색 결과 목록에 대한 가중치 리스트.

    Returns:
        융합된 검색 결과 목록. (문서 ID, 점수) 튜플의 리스트입니다.
    """

    document_scores = {}
    for i, ranked_list in enumerate(ranked_lists):
        weight = weights[i]
        for j, (doc_id, _) in enumerate(ranked_list):
            if doc_id not in document_scores:
                document_scores[doc_id] = 0
            document_scores[doc_id] += weight / (j + 1)  # 순위는 1부터 시작하므로 j + 1

    sorted_scores = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_scores

# 예시 검색 결과 목록 및 가중치
ranked_lists = [
    [("A", 1), ("B", 2), ("C", 3)],
    [("B", 1), ("A", 2), ("D", 3)],
    [("A", 1), ("C", 2), ("D", 3)],
]
weights = [0.5, 0.3, 0.2]  # 각 목록에 대한 가중치

# 가중 RRF 융합
fused_results = weighted_reciprocal_rank_fusion(ranked_lists, weights)

# 결과 출력
print(fused_results)

[('A', 0.8500000000000001), ('B', 0.55), ('C', 0.26666666666666666), ('D', 0.16666666666666666)]


In [111]:
def weighted_reciprocal_rank_fusion(ranked_lists):
    """
    여러 개의 검색 결과 목록을 RRF를 이용하여 융합합니다.
    The Score is reciprocal of rank

    Args:
        ranked_lists: 검색 결과 목록들의 리스트. 각 목록은 (문서 ID, 순위) 튜플의 리스트입니다.

    Returns:
        융합된 검색 결과 목록. (문서 ID, 점수) 튜플의 리스트입니다.
    """

    document_scores = {}
    for ranked_list in ranked_lists:
        for i, (doc, score) in enumerate(ranked_list):
            doc_id = doc.metadata['document_id']
            if doc_id not in document_scores:
                document_scores[doc_id] = 0
            document_scores[doc_id] += score / (i + 1)  # 순위는 1부터 시작하므로 i + 1

    sorted_scores = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_scores

In [112]:
ranked_lists = [converted_sparse_results, dense_results]

# RRF 융합
weighted_fused_results = weighted_reciprocal_rank_fusion(ranked_lists)

# 결과 출력
print(f"len of weighted_fused_results: {len(weighted_fused_results)}")
print(weighted_fused_results)

len of weighted_fused_results: 12
[('81f9053a-d441-4fdc-817b-f136662ef34e', 1.1030155221621196), ('cc8274bb-2c23-4802-bd80-de00c8862c13', 0.6882226467132568), ('036658bb-b498-4d1d-8078-0a7aab520669', 0.5419492909082855), ('d34f5126-dd3f-450a-a846-a82e943ef145', 0.4831059247090999), ('dcc58dfb-f428-4948-ae1e-90143779ae17', 0.3682329691864998), ('afa1f693-7e09-4109-a53b-fb098a3efa28', 0.2297805409952329), ('8d311550-6ba0-49f2-b8a4-fcd437d20fa0', 0.1875723480430343), ('30593878-327e-4d80-9187-e6be901eaea1', 0.18358894247469396), ('afa3c9d9-1b36-4b45-8695-8d33e24f6607', 0.16137972474098206), ('7eb0e53a-ee34-472b-b275-a0cb50a03d9f', 0.1008488563599492), ('8080929a-1cc7-4c63-890c-ffb72027edc6', 0.07633558057139408), ('519e4fc3-5fed-479b-878c-f8df59b1e945', 0.0)]


## 5. Generate  

Prompt  
https://python.langchain.com/api_reference/core/prompts.html#langchain-core-prompts  

### WITHOUT LangGraph  

### Define Model  

In [113]:
# %pip install -qU langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Load LLM model

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)


### Define Prompt and Chain  

기존에는 Topic 클래스의 description 항목에 간결한 설명이라고 나와있어서 길이가 짧았다.

```
# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")
```

In [114]:
# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="Detailed Explanation of the Topic")
    hashtags: str = Field(description="Hash tags format keywords (2 or more)")


# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
# 그러면 Topic 클래스의 템플릿에 맞는 형태로 JSON으로 반환.  
parser = JsonOutputParser(pydantic_object=Topic)


system_instruction = """
### Instruction ### 
Include details, pros, and cons

### Audience ###
Men in 30s who knows finance and investment in intermediate level
"""

# 질의 작성

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_instruction),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

### Question and Relevant Documents  

### Explain S&P 500 ETF  

In [115]:
question = "Explain S&P 500 ETF"


In [116]:
relevant_doc_ids = weighted_fused_results[0:5]
relevant_doc_ids = [doc_id for doc_id, score in relevant_doc_ids]
relevant_doc_ids

['81f9053a-d441-4fdc-817b-f136662ef34e',
 'cc8274bb-2c23-4802-bd80-de00c8862c13',
 '036658bb-b498-4d1d-8078-0a7aab520669',
 'd34f5126-dd3f-450a-a846-a82e943ef145',
 'dcc58dfb-f428-4948-ae1e-90143779ae17']

In [117]:
relevant_docs = [doc for doc,_ in dense_results if doc.metadata['document_id'] in relevant_doc_ids]
relevant_docs

[Document(id='93a3b156-cb61-484a-ab9e-4cd86657a692', metadata={'document_id': 'cc8274bb-2c23-4802-bd80-de00c8862c13', 'chunk_id': 0}, page_content='Vanguard®\nVanguard Total Stock Market ETF   |  VTI As of December 31, 2024 \nInvestment approach •Seeks to track the performance of the CRSP US Total Market Index. •Large-, mid-, and small-cap equity diversified across growth and value styles. •Employs apassively managed, index-sampling strategy. •The fund remains fully invested. •Low expenses minimize net tracking error. About the benchmark •The CRSP US Total Market Index represents approximately 100% of investable companies in the U.S. equity market. •The index is designed to accurately represent the U.S. equity market and deliver low turnover. Performance history Total returns  2 for period ended December 31, 2024    \nVTI (Inception 2001-05-24)  Quarter Year to date  1 year 3years 5years 10 years Since inception Net asset value (NAV) return 3 2.62% 23.75% 23.75% 7.88% 13.80% 12.50% 8.8

In [118]:
docs_content = "\n\n".join(doc.page_content for doc in relevant_docs)
to_markdown(docs_content)

> Vanguard®
> Vanguard Total Stock Market ETF   |  VTI As of December 31, 2024 
> Investment approach   *Seeks to track the performance of the CRSP US Total Market Index.   *Large-, mid-, and small-cap equity diversified across growth and value styles.   *Employs apassively managed, index-sampling strategy.   *The fund remains fully invested.   *Low expenses minimize net tracking error. About the benchmark   *The CRSP US Total Market Index represents approximately 100% of investable companies in the U.S. equity market.   *The index is designed to accurately represent the U.S. equity market and deliver low turnover. Performance history Total returns  2 for period ended December 31, 2024    
> VTI (Inception 2001-05-24)  Quarter Year to date  1 year 3years 5years 10 years Since inception Net asset value (NAV) return 3 2.62% 23.75% 23.75% 7.88% 13.80% 12.50% 8.89% Market price return 4 2.68 23.71 23.71 7.89 13.81 12.50 8.89 Spliced Total Stock Market Index 2.63 23.77 23.77 7.87 13.81 12.50 8.90 Dow Jones U.S. Total Stock Market Index (formerly known as the Dow Jones Wilshire 5000 Index) through April 22, 2005; MSCI US Broad Market Index through June 2, 2013; and CRSP US Total Market Index thereafter. The performance data shown represent past performance, which is not aguarantee of future results. Investment returns and principal value will fluctuate, so investors’ shares, when sold, may be worth more or less than their original cost. Current performance may be lower or higher than the performance data cited. For performance data current to the most recent month-end, visit our website at  vanguard.com/performance . The performance of an index is not an exact representation of any particular investment, as you cannot invest directly in an index. Investment Products: Not FDIC Insured   *No Bank Guarantee   *May Lose Value 
> Investment focus 
> Value
> Mid
> Large
> Small
> Blend
> Investment styl e
> Market capitalization
> Growth
> Central tendency Expected range of fund holdings Quick facts Benchmark CRSP US Total Market Index Expense ratio 1 0.03% Dividend schedule Quarterly ETF total net assets $456,499 million Fund total net assets $1,777,963 million Inception date 2001-05-24 
> Trading information Ticker symbol VTI CUSIP number 922908769 IIV (intra-day ticker) VTI.IV Index ticker (Bloomberg) CRSPTMT Exchange NYSE Arca 
> 1.  As reported in the most recent prospectus. Afund’s current expense ratio may be lower or higher than the figure reported in the prospectus.  2.  Figures for periods of less than one year are cumulative returns. All other figures represent average annual returns. Fund performance figures assume the reinvestment of dividends and capital gains distributions; the figures are pre-tax and net of expenses. The above widely used comparative index represents unmanaged or average returns on various financial assets that can be compared with the fund’s total returns for the purpose of measuring relative performance.  3.  As of 4p.m., Eastern time, when the regular trading session of the New York Stock Exchange typically closes.  4.  Effective July 15, 2024, the market price returns are calculated using the official closing price as reported by the ETF’s primary exchange. Prior to July 15, 2024, the market price returns were calculated using the midpoint between the bid and ask prices as of the closing time of the New York Stock Exchange (typically 4p.m., Eastern time). The returns shown do not represent the returns you would receive if you traded shares at other times.
> 
> 2025년01월31일
>  
>  
>  
> ※삼성자산운용주식회사는 이 투자신탁의 투자대상 자산의 종류 및 위험도를
> 감안하여 2등급으로 분류하였습니다. 펀드의 위험 등급은 운용실적, 시장 상황
> 등에 따라 변경될 수 있다는 점을 유의하여 투자판단을 하시기 바랍니다.
> 2025년 01월 31일 기준
> Kodex 미국S&P500
>  
>  (379800)
> S&P500 Total Return Index(KRW)를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운
> 용하여 투자대상 자산의 가치상승에 따른 수익을 추구합니다.
> - 미국 증권시장 상장 대형주 500여개 종목에 투자하는 미국 대표지수 ETF
> - 배당 재투자를 통한 장기 복리 효과 기대
> 기본정보
> ETF명 Kodex 미국S&P500
> 거래소코드 379800
> 기초지수 S&P 500 Total Return Index
> ETF순자산총액       37,917.08억원
> 1주당 NAV       20,157.94원
> 총 보수 연 0.0099%
> (지정판매 0.001%, 집합투자 0.0009%, 신탁 0.005%, 일반사무 0.003%)
> 상장일 2021.04.09
> 분배금 미지급(발생 시 재투자)
> 운용회사 삼성자산운용
> 사무수탁회사 신한펀드파트너스
> 수탁은행 한국씨티은행
> 환매방법 유가증권 시장을 통한 매도, 지정참가회사를 통한 해지에 의한 환매
> 설정단위 50,000주
>  
> 본 자료는 펀드의 단순 정보제공을 위해 작성된 것으로써, 본 펀드의 투자광고 및 투자권유를 위해 작성된 자료가 아닙니다. 따라서 본 자료는 삼성자산운용 홈페이지 게시 외에는 본 펀드에 가입하지 않은 고객에게 투자광고 또는 투자권유의
> 목적으로 제시되거나 제공될 수 없습니다. 본 자료는 신뢰할 만한 정보를 토대로 작성된 것이나 그 정확성이나 완전성에 대해 삼성자산운용은 어떠한 보장도 하지 않습니다. 집합투자증권은 예금자보호법에 따라 예금보험공사가 보호하지 않
> 습니다. 집합투자증권은 자산가격 변동, 환율변동, 신용등급 하락 등에 따른 원금손실(0~100%)이 발생할 수 있으며, 원금손실 발생 시 투자자에게 귀속됩니다. 투자자는 집합투자증권에 대하여 금융상품 판매업자로부터 충분한 설명을 받을
> 권리가 있으며, 투자 전(간이) 투자설명서 및 집합투자규약을 반드시 읽어보시기 바랍니다. 과거의 실적이 미래의 수익을 보장하는 것은 아닙니다.
> ※준법감시인 승인필 202501-2ETFE4
> 지급기준일 금액(원)
> -
> 분배금 지급현황
> 지급기준일 금액(원)
> -
> ※ 증권거래비용 등이 추가로 발생할 수 있습니다.
> 종목명 비중(%)
> NVIDIA Corp 7.00
> APPLE Inc 6.56
> MICROSOFT 6.44
> Amazon.com Inc 4.27
> Meta Platforms Inc-CL A 2.69
> 상위 10종목(%)
> 종목명 비중(%)
> ALPHABET INC-CL A 2.24
> TESLA MOTORS 2.23
> BROADCOM LTD 2.18
> ALPHABET INC-CL C 1.84
> BERKSHIRE HATHAWAY
> CL B ORD. 1.63
> 수익률 그래프(%)
> ※  분배금 재투자를 가정한 세전수익률 기준입니다.
> 21.04.09 22.03.23 23.03.06 24.02.17 25.01.31
> ETF 기초지수
> 기간 수익률(%)
>  1M 3M 6M 1Y 연초이후 상장이후
> ETF 0.37 10.15 18.42 35.49 0.37 101.60
> 기초지수 0.40 10.30 18.65 36.03 0.40 104.95
> 기초지수 대비 -0.03 -0.15 -0.23 -0.54 -0.03 -3.35
> ※ 상기 수익률은 세전수익률로, 과거의 운용실적이 미래의 수익을 보장하는 것은 아닙니다.
> 섹터 비중(%)
> IT 32.51
> 금융 10.94
> 임의소비재 10.90
> 헬스케어 9.34
> 커뮤니케이션 8.84
> 업종비중(%)
> ※ 향후 시장상황에 따라 달라질 수 있습니다.
> ※ 위 비중은 주식 주요 업종 구성 비율입니다.
> 
> AUGUST 1, 2024
> 2024 Prospectus
> iShares Trust
>   * iShares Core S&P 500 ETF | IVV | NYSE ARCA
> The Securities and Exchange Commission (“SEC”) has not approved or disapproved
> these securities or passed upon the adequacy of this prospectus. Any representation to
> the contrary is a criminal offense.
> 
> IVV
> iShares Core S&P 500 ETF
> Fact Sheet as of 31-Dec-2024
> The iShares Core S&P 500 ETF seeks to track the investment results of an index 
> composed of large-capitalization U.S. equities.
> WHY IVV?
> 1 Exposure to large, established U.S. companies
> 2 Low cost, tax efficient access to 500 of the largest cap U.S. stocks
> 3 Use at the core of your portfolio to seek long-term growth
> GROWTH OF HYPOTHETICAL 10,000 USD SINCE INCEPTION
> Fund  Benchmark  
> The Growth of $10,000 chart reflects a hypothetical $10,000 investment and assumes 
> reinvestment of dividends and capital gains. Fund expenses, including management fees and 
> other expenses were deducted.
> PERFORMANCE
> 1 Year 3 Year 5 Year 10 Year Since Inception
> NAV 24.98% 8.91% 14.49% 13.06% 7.79%
> Market Price 24.94% 8.91% 14.51% 13.06% 7.79%
> Benchmark 25.02% 8.94% 14.53% 13.10% 7.85%
> The  performance  quoted  represents  past  performance  and  does  not  guarantee  future
> results.  Investment  return  and  principal  value  of  an  investment  will  fluctuate  so  that  an
> investor’s  shares,  when  sold  or  redeemed,  may  be  worth  more  or  less  than  the  original
> cost.  Current  performance  may  be  lower  or  higher  than  the  performance  quoted.
> Performance  data  current  to  the  most  recent  month  end  may  be  obtained  by  visiting  
> www.iShares.com or www.blackrock.com.
> Beginning 8/10/20, the market price returns are calculated using the closing price. 
> Prior to 8/10/20, the market price returns were calculated using the midpoint of the bid/ask spread
> at 4:00 PM ET. The returns shown do not represent the returns you would receive if you traded
> shares at other times.
> KEY FACTS
> Fund Launch Date 05/15/2000
> Benchmark S&P 500 Index (USD)
> 30 Day SEC Yield 1.22%
> Number of Holdings 503
> Net Assets $585,743,415,248
> Ticker IVV
> CUSIP 464287200
> Exchange NYSE Arca
> TOP HOLDINGS (%)
> APPLE INC 7.58
> NVIDIA CORP 6.60
> MICROSOFT CORP 6.28
> AMAZON COM INC 4.11
> META PLATFORMS INC CLASS 
> A 2.56
> TESLA INC 2.26
> ALPHABET INC CLASS A 2.22
> BROADCOM INC 2.17
> ALPHABET INC CLASS C 1.82
> BERKSHIRE HATHAWAY INC 
> CLASS B 1.66
> 37.26
> Holdings are subject to change.
> 
> SPDR S&P 500 
> ETF Trust (SPY) 
> Delivering Unrivaled 
> Liquidity to Investors

### Vanilla Generation  

In [119]:
answer_vanilla = chain.invoke({"question": question})
answer_vanilla

{'description': "The S&P 500 ETF is an exchange-traded fund (ETF) that tracks the S&P 500 index, a market-capitalization-weighted index of the 500 largest publicly traded companies in the U.S.  Investing in an S&P 500 ETF means you're essentially buying a small piece of all these companies, providing broad diversification across various sectors of the American economy.  These ETFs aim to mirror the index's performance, offering a convenient way to gain exposure to the large-cap U.S. equity market.\n\n**Details:**\n\n* **Underlying Asset:** The S&P 500 index, representing 500 of the largest U.S. companies.\n* **Trading:** Traded on major stock exchanges like any other stock, offering intraday liquidity.\n* **Management:** Typically passively managed, meaning they aim to replicate the index, minimizing management fees.\n* **Dividends:**  Distribute dividends received from the underlying companies in the index.\n* **Expense Ratio:**  A small annual fee charged to cover administrative cost

In [120]:
to_markdown(answer_vanilla['description'])

> The S&P 500 ETF is an exchange-traded fund (ETF) that tracks the S&P 500 index, a market-capitalization-weighted index of the 500 largest publicly traded companies in the U.S.  Investing in an S&P 500 ETF means you're essentially buying a small piece of all these companies, providing broad diversification across various sectors of the American economy.  These ETFs aim to mirror the index's performance, offering a convenient way to gain exposure to the large-cap U.S. equity market.
> 
> **Details:**
> 
> * **Underlying Asset:** The S&P 500 index, representing 500 of the largest U.S. companies.
> * **Trading:** Traded on major stock exchanges like any other stock, offering intraday liquidity.
> * **Management:** Typically passively managed, meaning they aim to replicate the index, minimizing management fees.
> * **Dividends:**  Distribute dividends received from the underlying companies in the index.
> * **Expense Ratio:**  A small annual fee charged to cover administrative costs; typically very low for S&P 500 ETFs.
> * **Examples:** IVV (iShares Core S&P 500), VOO (Vanguard S&P 500 ETF), SPY (SPDR S&P 500 ETF Trust)
> 
> **Pros:**
> 
> * **Diversification:** Instant diversification across multiple sectors and companies, reducing risk compared to individual stock picking.
> * **Low Cost:**  Expense ratios are generally very low, making them a cost-effective investment.
> * **Liquidity:** Highly liquid, allowing for easy buying and selling throughout the trading day.
> * **Strong Historical Performance:** The S&P 500 has historically delivered strong long-term returns.
> * **Transparency:** Holdings are readily available and updated regularly.
> * **Simplicity:** Easy to understand and manage, making it suitable for both beginners and experienced investors.
> 
> **Cons:**
> 
> * **Market Risk:**  Performance is tied to the overall U.S. market, meaning potential for losses during market downturns.
> * **Limited Upside Potential:**  While offering steady growth, potential returns may be capped compared to more actively managed funds or individual stock picking.
> * **No Control Over Holdings:** You're invested in all 500 companies, including those you might not individually choose.
> * **Concentration Risk:** While diversified across sectors, the index is still concentrated in U.S. large-cap equities.  This can be a disadvantage if other market segments outperform.
> * **Dividend Taxation:** Dividends received are taxable, which can impact overall returns.

### Modify Prompt to HAVE context argument  

In [ ]:
# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="Detailed Explanation of the Topic")
    hashtags: str = Field(description="Hash tags format keywords (2 or more)")


# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
# 그러면 Topic 클래스의 템플릿에 맞는 형태로 JSON으로 반환.  
parser = JsonOutputParser(pydantic_object=Topic)


system_instruction = """
### Instruction ### 
Include details, pros, and cons
If relevant documents exist, include them in the context below.
{context}

### Audience ###
Men in 30s who knows finance and investment in intermediate level
"""

# 질의 작성

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_instruction),
        ("human", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

### RAG  

In [ ]:

answer_chain = chain.invoke({"question": question, "context": docs_content})
to_markdown(answer_chain['description'])

> SPY, the SPDR S&P 500 ETF Trust, is one of the most popular and liquid ETFs in the world, tracking the S&P 500 index.  This means it aims to mirror the performance of the 500 largest publicly traded companies in the U.S., offering broad market exposure across all eleven GICS sectors.  Launched in 1993, it was the first-ever ETF listed in the United States and remains a cornerstone of many investment portfolios.
> 
> **Details:**
> * **Objective:** Tracks the S&P 500 Index.
> * **Holdings:**  503 large-cap U.S. equities, weighted by market capitalization.
> * **Expense Ratio:** 0.0945% (SPY) vs. 0.02% (SPLG, a similar S&P 500 ETF)
> * **Trading:** Highly liquid, trades on major exchanges.
> * **Dividends:** Distributed quarterly.
> 
> **Pros:**
> * **Diversification:** Provides instant diversification across the largest U.S. companies.
> * **Low Cost:**  Relatively low expense ratio, though alternatives like SPLG offer even lower costs.
> * **Liquidity:**  Easy to buy and sell due to high trading volume.
> * **Transparency:** Holdings are publicly disclosed daily.
> * **Long Track Record:**  Decades of historical performance data available.
> 
> **Cons:**
> * **Concentration Risk:** Performance is tied to the U.S. large-cap market, so it's not immune to market downturns.
> * **No Outperformance:** Designed to match the index, not beat it.  Actively managed funds might offer higher returns (but also higher risk).
> * **Expense Ratio (relative):** While low compared to many mutual funds, other S&P 500 ETFs like SPLG have lower expense ratios, which can make a difference over the long term.
> * **Limited Exposure:** Focuses solely on large-cap U.S. equities, missing out on potential opportunities in small-cap, international, or other asset classes.

### Explain KODEX S&P 500 ETF  


In [130]:
question = "Explain KODEX S&P 500 ETF"

sparse_results = bm25_retriever_modified.invoke(question)
dense_results = vector_store.similarity_search_with_score(question, k=10)
converted_sparse_results = convert_sparse_score(sparse_results)
weighted_fused_results = weighted_reciprocal_rank_fusion([converted_sparse_results, dense_results])
relevant_doc_ids = weighted_fused_results[0:5]
relevant_doc_ids = [doc_id for doc_id, score in relevant_doc_ids]
relevant_docs = [doc for doc,_ in dense_results if doc.metadata['document_id'] in relevant_doc_ids]
docs_content = "\n\n".join(doc.page_content for doc in relevant_docs)
to_markdown(docs_content)

> 삼성 KODEX 미국S&P500 증권상장지수투자신탁[주식] 
> 신탁계약서 
>  
>  
>  
>  
>  
>  
>  
>  
>  
>  
>  
>  
> 집합투자업자: 삼성자산운용주식회사 
> 신 탁 업 자 : 주식회사한국씨티은행
> 
> 1
>  <간이투자설명서> (작성기준일 : 2025.01.20)
> 삼성KODEX 미국S&P500 증권상장지수투자신탁[주식](펀드 코드: DF732)
> 투자 위험 등급
> 2등급(높은위험)
> 삼성자산운용주식회사는 이 투자신탁의 일간 수익률의 최대손실예상액을 감안
> 하여 2등급으로 분류하였습니다.
> 1 2 3 4 5 6 집합투자증권은 『예금자보호법』에 따라 보호되지 않는 실적배당상품입니다.
> 해당 집합투자기구는 해외주식을 주된 투자대상으로 하여 신탁재산의 60% 이상을 투자하며, 미국거래
> 소에 상장된 주식 투자에 따른 특정국가 집중투자위험, 해외투자에 따른 환율변동위험, 기초지수 추종
> 에 따른 추적오차 발생위험 등이 있으므로 투자에 신중을 기하여 주시기 바랍니다.
> 매우
> 높은
> 위험
> 높은
> 위험
> 다소
> 높은
> 위험
> 보통
> 위험
> 낮은
> 위험
> 매우
> 낮은
> 위험
> 이 요약정보는 삼성KODEX 미국S&P500 증권상장지수투자신탁[주식]의 투자설명서의 내용 중 중요사항을 발췌·요약한 핵심정보를 담고 있습니다. 따라서 자세
> 한 정보가 필요하신 경우에는 동 집합투자증권을 매입하기 이전에 투자설명서를 반드시 참고하시기 바랍니다.
> [요약정보]
> 투자목적
> 및
> 투자전략
> 이 투자신탁은 미국거래소에 상장된 주식을 주된 투자대상으로 하여 신탁재산의 60% 이상을 투자하며, S&P Dow Jones Indices에서 산출·발표하
> 는 S&P500 TR Index(KRW)지수를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운용할 계획입
> 니다.
> 분류 투자신탁, 증권(주식형), 개방형(중도환매가능), 추가형(추가납입가능), 상장지수투자신탁
> 투자비용
> 명칭
> 투자자가 부담하는 수수료, 총보수 및 비용 1,000만원 투자시 투자자가 부담하는 투자기간
> 별 총보수ㆍ비용 예시 (단위 : 천원)
> 판매수수료 총보수 동종유형
> 총보수
> 총 보수
> ㆍ비용 1년 2년 3년 5년 10년판매보수
> 삼성KODEX 미국S&P500 증권
> 상장지수투자신탁[주식] 없음  0.0099%  0.001%  0.0925%      9     19     29     50    114
> (주1) 구체적인 투자비용은 투자설명서' 제2부. 집합투자기구에 관한 사항 중 13. 보수 및 수수료에 관한 사항'을 참고하시기 바랍니다.
> (주2) '동종유형 총보수'는 한국금융투자협회에서 공시하는 동종유형 집합투자기구 전체의 평균 총보수비용을 의미합니다.
> (주3) '1,000만원 투자시 투자자가 부담하는 투자기간별 총비용 예시'는 투자자가 1,000만원을 투자했을 경우 향후 투자기간별 지불하게 되는 총비용 (판매수수료 +
> 총보수비용)을 의미합니다. 선취판매수수료 및 총보수·비용은 일정하고, 이익금은 모두 재투자하며, 연간 투자수익률은 5%로 가정하되, 기타비용(증권거래 비
> 용 및 금융비용 제외)의 변동, 보수의 인상 또는 이하 여부 등에 따라 실제 부담하게 되는 보수 및 비용이 달라질 수 있습니다.
> 투자실적
> 추이
> (연평균
> 수익률)
> 최근 1년 최근 2년 최근 3년 최근 5년
> 24/01/21~
> 25/01/20
> 23/01/21~
> 25/01/20
> 22/01/21~
> 25/01/20 
> 삼성KODEX 미국S&P500 증권상
> 장지수투자신탁[주식] 2021-04-07   37.22   35.92   18.59   19.87
> 비교지수(%) 2021-04-07   37.74   36.41   19.03   20.40
> 수익률 변동성(%) 2021-04-07   14.11   13.19   15.59   14.85
> 명칭 최초설정일
> - 비교지수 : S&P500 TR Index(KRW) * 100%(비교지수 성과에는 투자신탁에 부과되는 보수 및 비용이 반영되지 않음)
> - 연평균 수익률은 해당 기간동안의 누적수익률을 기하평균방식으로 계산한 것으로 집합투자기구 총비용 지급후 해당기간동안의 세전평균 수익률을 나타내는 수
> 치입니다.
> - 수익률 변동성(표준편차)은 해당기간 펀드의 연환산 주간수익률이 평균수익률에서 통상적으로 얼마만큼 등락했는지를 보여주는 수치로서, 변동성이 높을수록
> 수익률 등락이 빈번해 펀드의 손실위험이 높다는 것을 의미합니다.
> 설정일이후
> 
> 2025년01월31일
>  
>  
>  
> ※삼성자산운용주식회사는 이 투자신탁의 투자대상 자산의 종류 및 위험도를
> 감안하여 2등급으로 분류하였습니다. 펀드의 위험 등급은 운용실적, 시장 상황
> 등에 따라 변경될 수 있다는 점을 유의하여 투자판단을 하시기 바랍니다.
> 2025년 01월 31일 기준
> Kodex 미국S&P500
>  
>  (379800)
> S&P500 Total Return Index(KRW)를 기초지수로 하여 1좌당 순자산가치의 변동률을 기초지수의 변동률과 유사하도록 투자신탁재산을 운
> 용하여 투자대상 자산의 가치상승에 따른 수익을 추구합니다.
> - 미국 증권시장 상장 대형주 500여개 종목에 투자하는 미국 대표지수 ETF
> - 배당 재투자를 통한 장기 복리 효과 기대
> 기본정보
> ETF명 Kodex 미국S&P500
> 거래소코드 379800
> 기초지수 S&P 500 Total Return Index
> ETF순자산총액       37,917.08억원
> 1주당 NAV       20,157.94원
> 총 보수 연 0.0099%
> (지정판매 0.001%, 집합투자 0.0009%, 신탁 0.005%, 일반사무 0.003%)
> 상장일 2021.04.09
> 분배금 미지급(발생 시 재투자)
> 운용회사 삼성자산운용
> 사무수탁회사 신한펀드파트너스
> 수탁은행 한국씨티은행
> 환매방법 유가증권 시장을 통한 매도, 지정참가회사를 통한 해지에 의한 환매
> 설정단위 50,000주
>  
> 본 자료는 펀드의 단순 정보제공을 위해 작성된 것으로써, 본 펀드의 투자광고 및 투자권유를 위해 작성된 자료가 아닙니다. 따라서 본 자료는 삼성자산운용 홈페이지 게시 외에는 본 펀드에 가입하지 않은 고객에게 투자광고 또는 투자권유의
> 목적으로 제시되거나 제공될 수 없습니다. 본 자료는 신뢰할 만한 정보를 토대로 작성된 것이나 그 정확성이나 완전성에 대해 삼성자산운용은 어떠한 보장도 하지 않습니다. 집합투자증권은 예금자보호법에 따라 예금보험공사가 보호하지 않
> 습니다. 집합투자증권은 자산가격 변동, 환율변동, 신용등급 하락 등에 따른 원금손실(0~100%)이 발생할 수 있으며, 원금손실 발생 시 투자자에게 귀속됩니다. 투자자는 집합투자증권에 대하여 금융상품 판매업자로부터 충분한 설명을 받을
> 권리가 있으며, 투자 전(간이) 투자설명서 및 집합투자규약을 반드시 읽어보시기 바랍니다. 과거의 실적이 미래의 수익을 보장하는 것은 아닙니다.
> ※준법감시인 승인필 202501-2ETFE4
> 지급기준일 금액(원)
> -
> 분배금 지급현황
> 지급기준일 금액(원)
> -
> ※ 증권거래비용 등이 추가로 발생할 수 있습니다.
> 종목명 비중(%)
> NVIDIA Corp 7.00
> APPLE Inc 6.56
> MICROSOFT 6.44
> Amazon.com Inc 4.27
> Meta Platforms Inc-CL A 2.69
> 상위 10종목(%)
> 종목명 비중(%)
> ALPHABET INC-CL A 2.24
> TESLA MOTORS 2.23
> BROADCOM LTD 2.18
> ALPHABET INC-CL C 1.84
> BERKSHIRE HATHAWAY
> CL B ORD. 1.63
> 수익률 그래프(%)
> ※  분배금 재투자를 가정한 세전수익률 기준입니다.
> 21.04.09 22.03.23 23.03.06 24.02.17 25.01.31
> ETF 기초지수
> 기간 수익률(%)
>  1M 3M 6M 1Y 연초이후 상장이후
> ETF 0.37 10.15 18.42 35.49 0.37 101.60
> 기초지수 0.40 10.30 18.65 36.03 0.40 104.95
> 기초지수 대비 -0.03 -0.15 -0.23 -0.54 -0.03 -3.35
> ※ 상기 수익률은 세전수익률로, 과거의 운용실적이 미래의 수익을 보장하는 것은 아닙니다.
> 섹터 비중(%)
> IT 32.51
> 금융 10.94
> 임의소비재 10.90
> 헬스케어 9.34
> 커뮤니케이션 8.84
> 업종비중(%)
> ※ 향후 시장상황에 따라 달라질 수 있습니다.
> ※ 위 비중은 주식 주요 업종 구성 비율입니다.
> 
> SPDR S&P 500 
> ETF Trust (SPY) 
> Delivering Unrivaled 
> Liquidity to Investors
> 
> 투자 위험 등급
> 2등급(높은위험) 삼성자산운용주식회사는 이 투자신탁의 일간 수익률의 최대손실예상액
> 을 감안하여 2등급으로 분류하였습니다. 펀드의 위험 등급은 운용실적, 시장
> 상황 등에 따라 변경될 수 있다는 점을 유의하여 투자판단을 하시기 바랍니다
> .
> 1 2 3 4 5 6
> 매우
> 높은
> 위험
> 높은
> 위험
> 다소
> 높은
> 위험
> 보통
> 위험
> 낮은
> 위험
> 매우
> 낮은
> 위험
> 투자설명서
>  이 투자설명서는 삼성KODEX 미국S&P500 증권상장지수투자신탁[주식]에 대한 자세한 내용을 담고 있습
> 니다. 따라서 삼성KODEX 미국S&P500 증권상장지수투자신탁[주식]를 매입하기 전에 반드시 이 투자설
> 명서를 읽어보시기 바랍니다.
> : 삼성KODEX 미국S&P500 증권상장지수투자신탁[주식] 1. 집합투자기구 명칭
>  2. 집합투자업자 명칭 : 삼성자산운용주식회사 
>  3. 판  매  회  사 : 집합투자업자(http://www.samsungfund.com) 및 금융투자협회(www.kofia.or.kr) 홈페이
> 지 참조
>  4. 작 성 기 준 일 : 2025. 1.20 
>  5. 증권신고서 효력발생일 : 2025. 1.24 
>  6. 모집(매출) 증권의 종류 및 수 : 투자신탁 수익증권 
>  [모집(매출) 총액 : 추가로 설정할 수 있는 수익증권의 총좌수는 10조좌입니다.] 
>  7. 모집(매출) 기간(판매기간)    : 2021년 3월 31일부터 모집을 개시하며 모집개시일 이후 특별한
> 사정이 없는 한 계속하여 모집할 수 있습니다.
>  8. 집합투자증권신고서 및 투자설명서의 열람장소 
>  가. 집합투자증권 신고서 
>  전자문서 : 금융위(금감원) 전자공시시스템 → http://dart.fss.or.kr 
>  나. 투자설명서 
>  전자문서 : 금융위(금감원) 전자공시시스템 → http://dart.fss.or.kr 
>  서면문서 : 집합투자업자, 금융위원회, 각 판매회사, 한국거래소 
>  9. 안정조작 또는 시장조성 관련 
>  ※ 이 투자설명서는 효력발생일까지 증권신고서의 기재사항 중 일부가 변경될 수 있으며, 개방형 집
> 합투자증권인 경우 효력발생일 이후에도 변경될 수 있습니다.
>  금융위원회가 투자설명서의 기재사항이 진실 또는 정확하다는 것을 인정하거나
> 그 증권 가치를 보증 또는 승인하지 아니함을 유의하시기 바랍니다. 또한 이 집
> 합투자증권은 "예금자보호법"에 따라 보호되지 않는 실적배당 상품으로 투자원금
> 의 손실이 발생할 수 있으므로 투자에 신중을 기하여 주시기 바랍니다.

### Vanilla Generation  

In [131]:
# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="Detailed Explanation of the Topic")
    hashtags: str = Field(description="Hash tags format keywords (2 or more)")


# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
# 그러면 Topic 클래스의 템플릿에 맞는 형태로 JSON으로 반환.  
parser = JsonOutputParser(pydantic_object=Topic)


system_instruction = """
### Instruction ### 
Include details, pros, and cons

### Audience ###
Men in 30s who knows finance and investment in intermediate level
"""

# 질의 작성

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_instruction),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
answer_vanilla = chain.invoke({"question": question})
to_markdown(answer_vanilla['description'])

> The KODEX S&P 500 ETF is a Korean Exchange-Traded Fund (ETF) that aims to track the performance of the S&P 500 Index. This index represents 500 of the largest publicly traded companies in the U.S., covering approximately 80% of the total U.S. equity market capitalization.  Investing in this ETF offers a convenient way to gain broad exposure to the U.S. stock market without having to purchase individual stocks.  The fund achieves this by holding a portfolio of stocks that mirrors the composition of the S&P 500.  It's passively managed, meaning it aims to replicate the index rather than actively picking stocks.  Dividends received from the underlying stocks are typically reinvested.  This ETF is traded on the Korea Exchange (KRX) and denominated in Korean Won (KRW), although the underlying assets are priced in USD.  Therefore, fluctuations in the KRW/USD exchange rate will impact returns for Korean investors.
> 
> **Key Features for Investors in their 30s:**
> 
> * **Diversification:**  Provides instant diversification across a wide range of sectors and companies, reducing risk compared to holding individual stocks. This is particularly beneficial for those building a long-term portfolio.
> * **Long-term Growth Potential:** The S&P 500 has historically delivered strong long-term returns, making it a suitable investment for long-term goals like retirement.
> * **Cost-Effective:** ETFs generally have lower expense ratios than actively managed funds, maximizing your returns over time.
> * **Liquidity:**  KODEX S&P 500 ETF is highly liquid, meaning you can easily buy and sell shares on the KRX.
> * **Currency Risk:**  Returns are affected by fluctuations in the KRW/USD exchange rate. This can be a benefit or a drawback depending on currency movements.
> 
> **Pros:**
> 
> * **Broad Market Exposure:**  Covers a significant portion of the U.S. equity market.
> * **Simplicity:** Easy to understand and manage.
> * **Low Cost:**  Typically lower expense ratios than actively managed funds.
> * **Tax Efficiency:**  Generally more tax-efficient than mutual funds.
> 
> **Cons:**
> 
> * **No Outperformance Potential:**  Designed to match the index, not outperform it.
> * **Currency Risk:**  Fluctuations in the KRW/USD exchange rate can impact returns.
> * **U.S. Market Concentration:**  Your investment is concentrated in the U.S. market, limiting exposure to other global opportunities.
> * **Dividend Withholding Tax:** Dividends received from U.S. companies may be subject to withholding tax.

### RAG

In [132]:
# Define a output parser
# 원하는 결과값 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="Detailed Explanation of the Topic")
    hashtags: str = Field(description="Hash tags format keywords (2 or more)")


# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
# 그러면 Topic 클래스의 템플릿에 맞는 형태로 JSON으로 반환.  
parser = JsonOutputParser(pydantic_object=Topic)


system_instruction = """
### Instruction ### 
Include details, pros, and cons
If relevant documents exist, include them in the context below.
{context}

### Audience ###
Men in 30s who knows finance and investment in intermediate level
"""

# 질의 작성

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_instruction),
        ("human", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
answer_chain = chain.invoke({"question": question, "context": docs_content})
to_markdown(answer_chain['description'])

> The KODEX S&P 500 ETF (379800) is a South Korean exchange-traded fund (ETF) that tracks the S&P 500 Total Return Index (KRW).  This means it aims to replicate the performance of the S&P 500, including dividends reinvested in Korean Won.  It's designed to provide investors with broad exposure to the U.S. large-cap equity market by investing in the 500 largest companies listed on American exchanges. Key features and considerations include:
> 
> **Details:**
> 
> * **Issuer:** Samsung Asset Management
> * **Inception Date:** April 9, 2021
> * **Expense Ratio:** 0.0099% per year (very low)
> * **Dividend Policy:**  Dividends are reinvested (accumulating ETF)
> * **Trading:** Traded on the Korea Exchange (KRX)
> * **Liquidity:** Typically very high due to its popularity and large AUM
> * **Currency:** KRW (Korean Won)
> 
> **Pros:**
> 
> * **Diversification:** Provides instant diversification across 500 leading U.S. companies.
> * **Low Cost:**  The expense ratio is significantly lower than many actively managed funds.
> * **Easy Access:**  Traded like a stock on the KRX, making it easy to buy and sell.
> * **Transparency:**  Holdings are publicly disclosed daily.
> * **Tax Efficiency:**  Generally more tax-efficient than mutual funds due to the ETF structure.
> * **Long-Term Growth Potential:** Designed for long-term investors seeking exposure to the U.S. equity market.
> 
> **Cons:**
> 
> * **Market Risk:**  Performance is tied to the S&P 500, so if the U.S. market declines, the ETF will also decline.
> * **Currency Risk:**  For Korean investors, fluctuations in the KRW/USD exchange rate can impact returns.
> * **No Control Over Holdings:**  You're investing in the entire index, not individual stocks.
> * **Foreign Market Regulations:**  Subject to South Korean regulations regarding ETF investments.
> * **Tracking Error:** While minimal, there might be a slight difference between the ETF's performance and the underlying index.
> 
> **Important Note:** The provided documents are excerpts from marketing materials and simplified prospectuses.  Before investing, always consult the full official investment prospectus and seek professional financial advice.

Issuer에 대한 내용이나 간이투자설명서인 총보수인 0.0099% 내용도 추가되었다.  